# Thomas' 4 charts

In [385]:
# ==========================================
# CELL 1: THEME & DATA SETUP (Run this once)
# ==========================================
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- THOMAS'S THEME COLORS ---
COLORS = {
    'background': '#0a1929',  # Dark Blue Background
    'surface': '#132f4c',      # Lighter Blue Card
    'primary': '#fbbf24',      # Gold/Yellow
    'secondary': '#f59e0b',    # Orange
    'success': '#10b981',      # Green
    'danger': '#ef4444',       # Red
    'text_primary': '#f1f5f9', # White Text
    'text_secondary': '#94a3b8', # Grey Text
    'border': '#334155',       # Border Lines
    'grid': '#1e293b',          # Grid Lines
    'info': '#3b82f6',
}

# --- DATA LOADING ---
def load_thomas_data():
    try:
        df = pd.read_csv('../cleaned_data/master_dataset.csv')
    except:
        df = pd.read_csv('master_dataset.csv') # Fallback
        
    df = df.dropna(subset=['PERIOD', 'GPA', 'STUDENT ID'])
    
    # Feature Engineering
    df['Age_Group'] = pd.cut(df['AGE'], bins=[0, 25, 35, 45, 100], labels=['18-25', '26-35', '36-45', '46+'])
    df['Pass_Status'] = df['GPA'].apply(lambda x: 'Pass' if x >= 2.0 else 'Fail')
    df['ATTENDANCE'] = df['ATTENDANCE'].fillna(0)
    df['SELF-STUDY HRS'] = df['SELF-STUDY HRS'].fillna(0)
    
    # Risk Logic
    df['Initial_Risk'] = pd.cut(df['GPA'].where(df['PERIOD'] == 'Sem 1'),
                                 bins=[0, 2.5, 3.0, 4.0],
                                 labels=['High Risk', 'Medium Risk', 'Low Risk'])
    df['Initial_Risk'] = df.groupby('STUDENT ID')['Initial_Risk'].transform('first')
    
    df['Period_Clean'] = df['PERIOD'].str.replace('Sem ', 'Semester ')
    df['Course_Code'] = df['STUDENT ID'].str.extract(r'^(\d{4})-')[0]
    
    return df

df = load_thomas_data()
print("✅ Data & Theme Loaded.")

✅ Data & Theme Loaded.


## CHART 1: COURSE DIFFICULTY

In [386]:
# ==========================================
# CHART 1: COURSE DIFFICULTY (Fixed Style)
# ==========================================
def plot_bubble_chart(df):
    # Data Prep
    stats = df.groupby('Course_Code').agg({
        'GPA': 'mean',
        'Pass_Status': lambda x: (x == 'Fail').sum() / len(x) * 100,
        'STUDENT ID': 'nunique'
    }).reset_index()
    stats.columns = ['Course_Code', 'Avg_GPA', 'Failure_Rate', 'Enrollment']
    
    # Plotting
    fig = px.scatter(
        stats, x='Avg_GPA', y='Failure_Rate', size='Enrollment', color='Failure_Rate',
        text='Course_Code', hover_name='Course_Code',
        color_continuous_scale=[[0, COLORS['success']], [0.5, COLORS['primary']], [1, COLORS['danger']]],
        range_color=[0, 40] # Fix color range so red is actually red
    )
    
    # --- VISUAL STYLING (The Part Missing Before) ---
    fig.update_layout(
        plot_bgcolor=COLORS['background'],
        paper_bgcolor=COLORS['background'],
        font=dict(color=COLORS['text_primary'], family='Inter, sans-serif'),
        title="<b>📊 Course Difficulty Matrix</b>",
        xaxis=dict(title="Average GPA", gridcolor=COLORS['grid'], linecolor=COLORS['border']),
        yaxis=dict(title="Failure Rate (%)", gridcolor=COLORS['grid'], linecolor=COLORS['border']),
        height=500,
        showlegend=False
    )
    
    # Text & Markers
    fig.update_traces(
        textposition='top center',
        textfont=dict(size=11, color=COLORS['text_primary']),
        marker=dict(line=dict(width=1, color='white'), opacity=0.9)
    )
    
    # Quadrant Lines
    fig.add_hline(y=20, line_dash="dot", line_color=COLORS['text_secondary'])
    fig.add_vline(x=3.0, line_dash="dot", line_color=COLORS['text_secondary'])
    
    # Annotations
    fig.add_annotation(x=3.5, y=35, text="✅ LOW RISK", showarrow=False, font=dict(color=COLORS['success'], size=14, weight="bold"))
    fig.add_annotation(x=2.5, y=35, text="🔴 HIGH RISK", showarrow=False, font=dict(color=COLORS['danger'], size=14, weight="bold"))

    return fig

fig1 = plot_bubble_chart(df)
fig1.show()

### 📊 Chart 1: Course Difficulty Matrix - Insights

**Insight 1:** Courses 1101 and 5113 are positioned in the "Low Risk" quadrant (high GPA ~3.2-3.4, low failure rate ~5%), indicating strong overall student performance with minimal intervention needed.

**Insight 2:** Course 1102 presents the highest risk profile with both the lowest average GPA (~3.0) and highest failure rate (~5-8%), suggesting this course may require curriculum review or additional student support mechanisms.

**Insight 3:** Course enrollment varies significantly (bubble sizes show 5113 and 5112 have larger enrollments than 1102 and 2101), yet larger courses maintain better performance metrics, indicating scalability doesn't negatively impact quality.

## CHART 2: GPA Trajectory by Risk Level

In [387]:
# ==========================================
# CHART 2: RISK TRAJECTORY (Fixed Style)
# ==========================================
def plot_trajectory_chart(df):
    # Filter Data
    retained = df.groupby('STUDENT ID')['PERIOD'].nunique()
    valid_students = retained[retained >= 2].index
    d = df[df['STUDENT ID'].isin(valid_students) & (df['PERIOD'] != 'Sem 4')]
    
    fig = go.Figure()
    
    # Define Colors Map
    risk_colors = {'High Risk': COLORS['danger'], 'Medium Risk': COLORS['primary'], 'Low Risk': COLORS['success']}
    
    for risk in ['High Risk', 'Medium Risk', 'Low Risk']:
        subset = d[d['Initial_Risk'] == risk]
        if len(subset) > 0:
            trend = subset.groupby('Period_Clean')['GPA'].mean().reset_index()
            fig.add_trace(go.Scatter(
                x=trend['Period_Clean'], y=trend['GPA'],
                mode='lines+markers', name=risk,
                line=dict(color=risk_colors[risk], width=4),
                marker=dict(size=10, symbol='circle')
            ))

    # --- VISUAL STYLING ---
    fig.update_layout(
        plot_bgcolor=COLORS['background'],
        paper_bgcolor=COLORS['background'],
        font=dict(color=COLORS['text_primary'], family='Inter, sans-serif'),
        title="<b>📈 GPA Trajectory by Initial Risk</b>",
        xaxis=dict(gridcolor=COLORS['grid'], linecolor=COLORS['border']),
        yaxis=dict(title="Average GPA", range=[1.5, 4.0], gridcolor=COLORS['grid'], linecolor=COLORS['border']),
        height=500,
        legend=dict(orientation="h", y=1.1)
    )
    
    # Passing Line
    fig.add_hline(y=2.0, line_dash="dash", line_color=COLORS['text_secondary'], annotation_text="Pass (2.0)")
    
    return fig

fig2 = plot_trajectory_chart(df)
fig2.show()

### 📈 Chart 2: GPA Trajectory by Risk Level - Insights

**Insight 1:** All three risk groups show consistent upward GPA trends from Semester 1 to Semester 3, with Low Risk students improving from ~3.4 to ~3.7, demonstrating that early performance indicators persist but students generally improve over time.

**Insight 2:** The gap between High Risk and Low Risk students remains relatively constant (~0.8-1.0 GPA points) across semesters, suggesting that initial risk classification is a strong predictor of long-term performance and early intervention is critical.

**Insight 3:** Medium Risk students (orange line) show the steepest improvement trajectory, crossing above 3.0 GPA by Semester 2, indicating this group benefits most from continued enrollment and may be ideal targets for retention support programs.

## CHART 3: Risk Hotspot % Below 2.5 GPA

In [388]:
import plotly.graph_objects as go

# ==========================================
# CHART 3: AT-RISK HEATMAP (< 2.5 GPA)
# ==========================================
def plot_heatmap_at_risk(df):
    # 1. Filter for valid age groups (>= 5 students)
    age_counts = df.groupby('Age_Group').size()
    valid_ages = age_counts[age_counts >= 5].index
    d = df[df['Age_Group'].isin(valid_ages)]
    
    # 2. Calculate "At-Risk" Rate Matrix (GPA < 2.5)
    matrix = d.groupby(['Age_Group', 'Course_Code']).apply(
        lambda x: (x['GPA'] < 2.5).sum() / len(x) * 100 if len(x) > 0 else 0
    ).reset_index()
    
    pivot = matrix.pivot(index='Age_Group', columns='Course_Code', values=0).fillna(0)
    
    # 3. Plotting
    fig = go.Figure(data=go.Heatmap(
        z=pivot.values,
        x=pivot.columns,
        y=pivot.index,
        # --- NEW: Add text to the heatmap cells ---
        text=pivot.values,              # The values to display
        texttemplate="%{text:.1f}%",    # Format: 1 decimal place + % sign
        textfont={"size": 10},          # Optional: Adjust font size to fit
        # ------------------------------------------
        colorscale=[
            [0.0, COLORS['success']],   # 0% At Risk = Green
            [0.2, COLORS['success']],
            [0.2, COLORS['primary']],   # 20-40% = Yellow/Gold
            [0.4, COLORS['primary']],
            [0.4, COLORS['danger']],    # >40% = Red
            [1.0, COLORS['danger']]
        ],
        hovertemplate='Age: %{y}<br>Course: %{x}<br>At-Risk Rate: %{z:.1f}%<extra></extra>',
        xgap=2, ygap=2 
    ))

    # 4. Visual Styling
    fig.update_layout(
        plot_bgcolor=COLORS['background'],
        paper_bgcolor=COLORS['background'],
        font=dict(color=COLORS['text_primary'], family='Inter, sans-serif'),
        title="<b>🔥 At-Risk Heatmap (% GPA < 2.5)</b>",
        xaxis=dict(title="Course Code", tickangle=-45),
        yaxis=dict(title="Age Group"),
        height=500
    )
    
    return fig

# Run Test
fig3 = plot_heatmap_at_risk(df)
fig3.show()

C:\Users\User\AppData\Local\Temp\ipykernel_38176\67626232.py:8: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\User\AppData\Local\Temp\ipykernel_38176\67626232.py:13: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\User\AppData\Local\Temp\ipykernel_38176\67626232.py:13: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warn

### 🔥 Chart 3: Risk Hotspot Heatmap - Insights

**Insight 1:** The 46+ age group exhibits the highest concentration of at-risk students across multiple courses, with 50-60% falling below 2.5 GPA in courses 1102 and 5113, indicating older adult learners may need tailored support or flexible learning options.

**Insight 2:** Courses 1101 and 2101 show strong performance (mostly green) for the 26-35 age group with 0-10% below threshold, suggesting these courses are well-suited for mid-career professionals or the content aligns with their work experience.

**Insight 3:** Course 2102 shows universally low risk across all age groups (all green/teal), indicating either effective course design, appropriate difficulty level, or successful student selection/preparation for this certificate program.

## CHART 4: Attendance Threshold Impact

In [389]:
# ==========================================
# CHART 4: ATTENDANCE IMPACT (With Numbers)
# ==========================================
def plot_attendance_chart_v2(df, current_threshold=75, course=None, nationality=None):
    # 1. Filter Data
    d_filtered = df.copy()
    if course and course != 'all': 
        d_filtered = d_filtered[d_filtered['Course_Code'] == course]
    if nationality and nationality != 'all': 
        d_filtered = d_filtered[d_filtered['NATIONALITY_STATUS'] == nationality]
        
    # 2. Safety Check
    if len(d_filtered) == 0:
        fig = go.Figure()
        fig.update_layout(title="⚠️ No Data", plot_bgcolor=COLORS['background'], paper_bgcolor=COLORS['background'], font={'color': COLORS['text_primary']})
        return fig

    # 3. Calculate Stats per band
    stats = []
    for band in range(50, 101, 5):
        subset = d_filtered[d_filtered['ATTENDANCE'] >= band]
        if len(subset) > 0:
            pass_r = (subset['Pass_Status'] == 'Pass').sum() / len(subset) * 100
            gpa = subset['GPA'].mean()
        else:
            pass_r = 0; gpa = 0
        stats.append({'Threshold': f'{band}%+', 'Pass_Rate': pass_r, 'Avg_GPA': gpa})
    stats_df = pd.DataFrame(stats)
    
    # 4. Plot Setup (Dual Axis)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    # Trace 1: Bars (Pass Rate) - Blue/Info Color
    fig.add_trace(go.Bar(
        x=stats_df['Threshold'], y=stats_df['Pass_Rate'],
        name='Pass Rate', marker_color=COLORS.get('info', '#3b82f6'), opacity=0.8,
        # --- NEW: Add Numbers on Top ---
        text=stats_df['Pass_Rate'].apply(lambda x: f'{x:.1f}%'),
        textposition='outside',
        textfont=dict(color=COLORS['text_primary'], size=10),
        # -------------------------------
        hovertemplate='Pass Rate: %{y:.1f}%<extra></extra>'
    ), secondary_y=False)
    
    # Trace 2: Line (GPA) - Red/Danger Color with Diamond markers
    fig.add_trace(go.Scatter(
        x=stats_df['Threshold'], y=stats_df['Avg_GPA'],
        name='Avg GPA', mode='lines+markers+text', # Added text mode
        # --- NEW: Add Numbers for GPA too ---
        text=stats_df['Avg_GPA'].apply(lambda x: f'{x:.2f}'),
        textposition='top center',
        textfont=dict(color=COLORS['danger'], size=10, weight='bold'),
        # ------------------------------------
        line=dict(color=COLORS['danger'], width=3),
        marker=dict(size=10, symbol='diamond', color='white', line=dict(width=2, color=COLORS['danger'])),
        hovertemplate='GPA: %{y:.2f}<extra></extra>'
    ), secondary_y=True)

    # 5. The Vertical Line
    label_to_find = f'{current_threshold}%+'
    if label_to_find in stats_df['Threshold'].values:
        fig.add_shape(type="line", xref="x", yref="paper",
            x0=label_to_find, x1=label_to_find, y0=0, y1=1,
            line=dict(color="white", width=2, dash="dash"))
        fig.add_annotation(xref="x", yref="paper",
            x=label_to_find, y=1.15, text=f"Current: {current_threshold}%", 
            showarrow=False, font=dict(color="white", size=12, weight="bold"))

    # 6. Final Styling
    fig.update_layout(
        plot_bgcolor=COLORS['background'], paper_bgcolor=COLORS['background'],
        font=dict(color=COLORS['text_primary'], family='Inter, sans-serif'),
        title="<b>🎓 Attendance Impact Analysis</b>", height=500,
        legend=dict(orientation="h", y=1.02, x=0.5, xanchor='center'),
        margin=dict(t=100)
    )
    
    # Axis Ranges (Expanded to fit numbers)
    fig.update_yaxes(title_text="Pass Rate (%)", range=[0, 115], gridcolor=COLORS['grid'], secondary_y=False)
    fig.update_yaxes(title_text="Average GPA", range=[1.5, 4.3], showgrid=False, secondary_y=True)
    fig.update_xaxes(gridcolor=COLORS['grid'], title="Minimum Attendance Requirement")
    
    return fig

# Run Test
fig4 = plot_attendance_chart_v2(df, current_threshold=75)
fig4.show()

### 📅 Chart 4: Attendance Threshold Impact - Insights

**Insight 1:** Pass rates remain consistently high (96-99%) across all attendance thresholds from 50% to 100%, indicating that even students with moderate attendance (50-60%) can achieve passing grades, suggesting course design accommodates flexible learning.

**Insight 2:** Average GPA shows a clear positive correlation with attendance, increasing from ~3.1 at 50% attendance to ~3.5 at 100% attendance, demonstrating that while passing is achievable with lower attendance, academic excellence requires consistent engagement.

**Insight 3:** The current threshold of 75% (marked with dashed line) captures 420 students with a 99.0% pass rate and 3.19 average GPA, representing an optimal balance between accessibility and academic rigor without being overly restrictive.

# Lin Kai's 4 charts

In [390]:
# ==========================================
# CELL 1: LINGGER'S SETUP & DATA LOADING
# ==========================================
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np

# --- 1. LINGGER'S THEME COLORS (Blue/Teal) ---
COLORS = {
    'background': '#0a1929',          
    'surface': '#132f4c',              
    'card': '#1e3a5f',                 
    'primary': '#06b6d4',              # Cyan/Teal accent
    'secondary': '#0891b2',            
    'success': '#10b981',              
    'danger': '#ef4444',               
    'warning': '#f59e0b',              
    'info': '#3b82f6',                 
    'text_primary': '#f1f5f9',         
    'text_secondary': '#94a3b8',       
    'border': '#334155',               
    'grid': '#1e293b'                  
}

CHART_TEMPLATE = {
    'layout': {
        'paper_bgcolor': COLORS['background'],
        'plot_bgcolor': COLORS['surface'],
        'font': {'color': COLORS['text_primary'], 'family': 'Inter, sans-serif'},
        'xaxis': {
            'gridcolor': COLORS['grid'],
            'linecolor': COLORS['border'],
            'tickfont': {'color': COLORS['text_secondary']}
        },
        'yaxis': {
            'gridcolor': COLORS['grid'],
            'linecolor': COLORS['border'],
            'tickfont': {'color': COLORS['text_secondary']}
        },
        'hovermode': 'closest',
        'margin': {'l': 60, 'r': 40, 't': 60, 'b': 60}
    }
}

# --- 2. DATA LOADING FUNCTION ---
def load_lingger_data():
    try:
        df = pd.read_csv('../cleaned_data/master_dataset.csv')
    except:
        df = pd.read_csv('master_dataset.csv') # Fallback
    
    df = df.dropna(subset=['PERIOD', 'GPA', 'STUDENT ID'])
    
    # Age Groups
    df['Age_Group'] = pd.cut(df['AGE'], bins=[0, 25, 35, 45, 100], labels=['18-25', '26-35', '36-45', '46+'])
    
    # Pass/Fail Status
    df['Pass_Status'] = df['GPA'].apply(lambda x: 'Pass' if x >= 2.0 else 'Fail')
    
    # Fill Missing Values
    df['ATTENDANCE'] = df['ATTENDANCE'].fillna(0)
    df['SELF-STUDY HRS'] = df['SELF-STUDY HRS'].fillna(0)
    
    # Specific to Lingger: Fill support columns with median
    for col in ['TEACHING SUPPORT', 'COMPANY SUPPORT', 'FAMILY SUPPORT', 'COURSE RELEVANCE']:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
            
    # Course Code
    df['Course_Code'] = df['STUDENT ID'].str.extract(r'^(\d{4})-')[0]
    
    return df

# --- 3. EXECUTE LOAD ---
df = load_lingger_data()
print(f"✅ Lingger's Data Loaded: {df.shape[0]} rows")

✅ Lingger's Data Loaded: 505 rows


## Chart 1: Study Effort by Nationality (Boxplot)

In [391]:
# ==========================================
# CHART 1: NATIONALITY STUDY EFFORT
# ==========================================
def create_nationality_chart(df, selected_nationality=None):
    # Filter if needed
    d = df.copy() if selected_nationality is None else df[df['NATIONALITY_STATUS'] == selected_nationality]
    
    # Create Box Plot
    fig = px.box(d, x='NATIONALITY_STATUS', y='SELF-STUDY HRS', color='NATIONALITY_STATUS',
                 points='outliers',
                 color_discrete_map={'SG Citizen': '#3b82f6', 'SG PR': '#8b5cf6', 'Foreigner': '#06b6d4'},
                 category_orders={'NATIONALITY_STATUS': ['SG Citizen', 'SG PR', 'Foreigner']})
    
    # Calculate averages for annotation
    avg_by_nat = d.groupby('NATIONALITY_STATUS')['SELF-STUDY HRS'].mean()
    
    # Apply Theme
    layout = CHART_TEMPLATE['layout'].copy()
    layout.update({
        'title': {'text': '<b>Study Effort by Nationality Status</b>', 'x': 0.5, 'xanchor': 'center'},
        'xaxis_title': 'Nationality Status',
        'yaxis_title': 'Weekly Self-Study Hours',
        'showlegend': False,
        'height': 450
    })
    fig.update_layout(**layout)

    # --- ENHANCED ANNOTATIONS (Make it POP!) ---
    for nat, avg in avg_by_nat.items():
        fig.add_annotation(
            x=nat, 
            y=avg,
            text=f"<b>Avg: {avg:.1f}h</b>",  # HTML Bold tags
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor=COLORS['primary'],    # Cyan Arrow
            ax=40,
            ay=-30,
            # --- STYLING BOX ---
            font=dict(size=13, color='#ffffff'), # Larger White Text
            bgcolor=COLORS['card'],              # Dark Card Background
            bordercolor=COLORS['primary'],       # Cyan Border
            borderwidth=2,                       # Thicker Border
            borderpad=5,                         # More breathing room
            opacity=0.95
        )
    
    return fig

# --- TEST CHART 1 ---
fig1 = create_nationality_chart(df)
fig1.show()

**Purpose**
- To compare **weekly self-study hours** across nationality groups as a demographic lens on learning behaviour, and check whether certain groups systematically study more/less (useful for profiling, not stereotyping).

**Why this graph**
- A **boxplot** shows the **median, spread (IQR), and overlap** across groups, so we can judge whether differences are broad patterns or just driven by a few outliers.

**Insights**
- Foreign students show **higher typical self-study hours (~15h/week)** than SG Citizens and SG PR (**~12–13h/week**), suggesting a demographic difference in study effort.
- The **large overlap** between groups indicates nationality explains **some** variation, but it is **not a strong standalone driver** of study behaviour.
- Because the lower tail exists in every group, the more reliable at-risk signal is **very low self-study hours**, regardless of nationality status.
- Practical takeaway: use nationality as **context**, but target support based on **individual behaviour thresholds** (e.g., ≤10h/week) paired with GPA/attendance to avoid false positives.

## Chart 2: Support Factors Impact on GPA (Ratings 1–5)

In [392]:
# ==========================================
# CHART 2: SUPPORT FACTORS IMPACT
# ==========================================
def create_support_chart(df):
    fig = go.Figure()
    
    # Define factors and colors
    factors = {
        'TEACHING SUPPORT': {'name': 'Teaching Support', 'color': '#3b82f6'},
        'COMPANY SUPPORT': {'name': 'Company Support', 'color': '#8b5cf6'},
        'FAMILY SUPPORT': {'name': 'Family Support', 'color': '#ec4899'},
        'COURSE RELEVANCE': {'name': 'Course Relevance', 'color': '#06b6d4'}
    }
    
    # Loop to add traces
    for col, info in factors.items():
        if col in df.columns:
            # Calculate mean GPA per level (1-5)
            d = df.groupby(col)['GPA'].mean().reset_index()
            fig.add_trace(go.Scatter(
                x=d[col], y=d['GPA'],
                mode='lines+markers',
                name=info['name'],
                line=dict(color=info['color'], width=3),
                marker=dict(size=10)
            ))

    # Apply Theme
    layout = CHART_TEMPLATE['layout'].copy()
    layout.update({
        'title': '<b>Support Factors Impact on GPA</b>',
        'xaxis_title': 'Support Level (1=Low, 5=High)',
        'yaxis_title': 'Average GPA',
        'yaxis_range': [1.5, 4.0],
        'height': 450,
        'legend': dict(orientation="h", y=1.1, x=0.5, xanchor='center')
    })
    fig.update_layout(**layout)
    
    # Add Passing Line
    fig.add_hline(y=2.0, line_dash="dash", line_color=COLORS['danger'], annotation_text="Pass (2.0)")
    
    return fig

# --- TEST CHART 2 ---
fig2 = create_support_chart(df)
fig2.show()

**Purpose**
- To evaluate how **perceived support factors** (e.g., relevance, teaching, family, company) relate to **academic performance (GPA)**, and identify which low-rated factors are most useful for early risk screening.

**Why this graph**
- A line chart is suitable because ratings are **ordered levels (1–5)**, making it easy to see **where GPA shifts most** (jumps, plateaus, dips) across the scale.

**Insights**
- Across factors, GPA shows its **largest improvement from ratings 1–2 → 3**, suggesting that moving students out of the “low-support” zone is where performance shifts most.
- From **3 → 5**, GPA continues to trend upward but with **smaller gains**, indicating a **plateau / diminishing returns** at higher support levels.
- **Family Support** shows the sharpest low→mid lift, making low family support a strong **early warning indicator** for potential academic difficulty.
- **Course Relevance** increases most steadily from 1→5, suggesting perceived relevance/motivation is a more consistent factor linked to performance.
- **Teaching Support** and **Company Support** show a **dip at rating 2** followed by a clear rise at 3, implying a **non-linear** relationship (useful for screening, but noisier than relevance/family).
- Actionable profile: students rating any factor **1–2** should be prioritised for follow-up, but escalation should be confirmed using **attendance + study hours + current GPA** for accurate targeting.

## Chart 3: Attendance × Study Effort Matrix (Pass Rate, Avg GPA, Student Count)

In [393]:
# ==========================================
# CHART 3: COMPENSATION MATRIX (HEATMAP)
# ==========================================
def create_heatmap(df, view_mode='pass_fail'):
    # Prepare Bins
    df_analysis = df.copy()
    df_analysis['Att_Bin'] = pd.cut(df_analysis['ATTENDANCE'], 
                                    bins=[0, 60, 75, 85, 100], 
                                    labels=['<60%', '60-75%', '75-85%', '85%+'])
    df_analysis['Study_Bin'] = pd.cut(df_analysis['SELF-STUDY HRS'], 
                                      bins=[0, 5, 10, 15, 100], 
                                      labels=['0-5h', '5-10h', '10-15h', '15h+'])
    
    # Calculate Pass Rate Matrix
    passfail_matrix = df_analysis.groupby(['Study_Bin', 'Att_Bin']).apply(
        lambda x: (x['Pass_Status'] == 'Pass').sum() / len(x) * 100 if len(x) > 0 else 0
    ).reset_index()
    
    pivot = passfail_matrix.pivot(index='Study_Bin', columns='Att_Bin', values=0).fillna(0)
    
    # Create Heatmap
    fig = go.Figure(go.Heatmap(
        z=pivot.values,
        x=pivot.columns,
        y=pivot.index,
        colorscale=[[0, COLORS['danger']], [0.5, COLORS['warning']], [1, COLORS['success']]],
        text=pivot.values,
        texttemplate="%{z:.0f}%",
        textfont={"family": "Inter", "size": 14, "color": "white"},
        xgap=2, ygap=2
    ))
    
    # Apply Theme
    layout = CHART_TEMPLATE['layout'].copy()
    layout.update({
        'title': '<b>Compensation Matrix: Pass Rate %</b>',
        'xaxis_title': 'Attendance Level',
        'yaxis_title': 'Weekly Study Hours',
        'height': 450
    })
    fig.update_layout(**layout)
    
    return fig

# --- TEST CHART 3 ---
fig3 = create_heatmap(df)
fig3.show()

C:\Users\User\AppData\Local\Temp\ipykernel_38176\2289651231.py:15: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\User\AppData\Local\Temp\ipykernel_38176\2289651231.py:15: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



**Purpose**
- To identify the **highest-risk combinations** of learning behaviours and show how **attendance and study effort interact** to influence outcomes (pass rate and GPA), so interventions can be targeted to the most critical behaviour patterns.

**Why this graph**
- Heatmaps are ideal for spotting **clusters** quickly: you can see “risk zones” at a glance, while the **count heatmap** shows whether patterns are based on meaningful student numbers.

**Insights**
- The clearest high-risk zone is **<60% attendance + 0–5h study**, which has **0% pass** and the **lowest GPA (~1.85)** — this is the strongest evidence of an at-risk profile.
- A broader risk region appears when **attendance <75%** and **study ≤10h/week**, where GPA remains low (**~1.85–2.48**) and pass rates are weaker.
- Most students are concentrated in **85%+ attendance** with **≥10h/week study**, which also produces the **highest GPAs (~3.38–3.59)** and near-**100% pass** — these behaviours act as protective factors.
- The pattern suggests **attendance is a gatekeeper**: once attendance is stable, higher study effort is more consistently associated with stronger outcomes.
- Intervention targeting: prioritise students in the **low attendance + low study** region first (highest risk and largest potential improvement), then address single-factor risks (low attendance OR low study).

## Chart 4: Simulate Risk Scenarios — Who Fails the Attendance Rule? (By Age)

In [394]:
# ==========================================
# CHART 4: RISK SIMULATION (STACKED BAR)
# ==========================================
def create_age_risk_chart(df, threshold=75):
    fig = go.Figure()
    age_groups = ['46+', '36-45', '26-35', '18-25']
    
    risk_pcts, safe_pcts, risk_text, safe_text = [], [], [], []
    
    for age in age_groups:
        d = df[df['Age_Group'] == age]
        total = len(d)
        if total == 0:
            risk_pcts.append(0); safe_pcts.append(0)
            risk_text.append(""); safe_text.append("")
            continue
            
        at_risk = (d['ATTENDANCE'] < threshold).sum()
        safe = total - at_risk
        
        r_pct = (at_risk / total * 100)
        s_pct = (safe / total * 100)
        
        risk_pcts.append(r_pct)
        safe_pcts.append(s_pct)
        risk_text.append(f"<b>{r_pct:.1f}%</b>" if r_pct > 5 else "")
        safe_text.append(f"<b>{s_pct:.1f}%</b>")
    
    # Add Traces
    fig.add_trace(go.Bar(
        y=age_groups, x=risk_pcts, orientation='h', name='Fails Requirement',
        marker_color=COLORS['danger'], text=risk_text, textposition='auto'
    ))
    
    fig.add_trace(go.Bar(
        y=age_groups, x=safe_pcts, orientation='h', name='Meets Requirement',
        marker_color=COLORS['success'], text=safe_text, textposition='auto'
    ))

    # Apply Theme
    layout = CHART_TEMPLATE['layout'].copy()
    layout.update({
        'title': f'<b>Risk Simulation (Threshold: {threshold}%)</b>',
        'xaxis_title': 'Percentage of Group',
        'yaxis_title': 'Age Group',
        'barmode': 'stack',
        'height': 400,
        'legend': dict(orientation="h", y=1.1)
    })
    fig.update_layout(**layout)
    
    return fig

# --- TEST CHART 4 (Try changing threshold to 80 or 90) ---
fig4 = create_age_risk_chart(df, threshold=75)
fig4.show()

**Purpose**
- To translate an attendance threshold into a **screening view** that highlights which age groups are more likely to be flagged as at-risk under the rule, supporting targeted and realistic intervention planning.

**Why this graph**
- A percentage-by-group chart clearly communicates **who fails a policy threshold**, which is directly actionable for prioritising outreach and resource allocation.

**Insights**
- At a set threshold (e.g., **75% attendance**), the **percentage failing increases with age**, indicating older learners are more likely to breach attendance requirements.
- The progression **26–35 → 36–45 → 46+** flags older students as a key **attendance-risk demographic**, especially when paired with low study hours.
- This likely reflects practical constraints (work/family responsibilities), so the best interventions should focus on **attendance recovery support** rather than assuming low ability.
- Targeted action: prioritise **early check-ins and monitoring** for older students approaching the threshold, and offer flexible catch-up options (recorded materials, structured make-up tasks) to prevent failure.

# Thomas' dashboard setup

In [395]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output, State, Dash # Import Dash explicitly
import dash_bootstrap_components as dbc
from datetime import datetime
import numpy as np

# ... Copy your COLORS and CHART_TEMPLATE dictionaries here ...
COLORS = {
    'background': '#0a1929',
    'surface': '#132f4c',
    'card': '#1e3a5f',
    'primary': '#fbbf24',
    'secondary': '#f59e0b',
    'success': '#10b981',
    'danger': '#ef4444',
    'warning': '#f59e0b',
    'info': '#3b82f6',
    'text_primary': '#f1f5f9',
    'text_secondary': '#94a3b8',
    'border': '#334155',
    'grid': '#1e293b'
}

CHART_TEMPLATE = {
    'layout': {
        'paper_bgcolor': COLORS['background'],
        'plot_bgcolor': COLORS['surface'],
        'font': {'color': COLORS['text_primary'], 'family': 'Inter, sans-serif'},
        'xaxis': {
            'gridcolor': COLORS['grid'],
            'linecolor': COLORS['border'],
            'tickfont': {'color': COLORS['text_secondary']}
        },
        'yaxis': {
            'gridcolor': COLORS['grid'],
            'linecolor': COLORS['border'],
            'tickfont': {'color': COLORS['text_secondary']}
        },
        'hovermode': 'closest',
        'margin': {'l': 60, 'r': 40, 't': 60, 'b': 60}
    }
}

In [396]:
# ... Copy the load_and_prepare_data() function here ...
def load_and_prepare_data():
    """Load and prepare the master dataset"""
    
    df = pd.read_csv('../cleaned_data/master_dataset.csv')
    df = df.dropna(subset=['PERIOD', 'GPA', 'STUDENT ID'])
    
    df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')
    df['COMMENCEMENT DATE'] = pd.to_datetime(df['COMMENCEMENT DATE'], errors='coerce')
    df['COMPLETION DATE'] = pd.to_datetime(df['COMPLETION DATE'], errors='coerce')
    
    df['Age_Group'] = pd.cut(df['AGE'], bins=[0, 25, 35, 45, 100], 
                              labels=['18-25', '26-35', '36-45', '46+'])
    
    df['Initial_Risk'] = pd.cut(df['GPA'].where(df['PERIOD'] == 'Sem 1'),
                                 bins=[0, 2.5, 3.0, 4.0],
                                 labels=['High Risk', 'Medium Risk', 'Low Risk'])
    df['Initial_Risk'] = df.groupby('STUDENT ID')['Initial_Risk'].transform('first')
    
    df['Pass_Status'] = df['GPA'].apply(lambda x: 'Pass' if x >= 2.0 else 'Fail')
    df['ATTENDANCE'] = df['ATTENDANCE'].fillna(0)
    df['SELF-STUDY HRS'] = df['SELF-STUDY HRS'].fillna(0)
    
    df['Period_Clean'] = df['PERIOD'].str.replace('Sem ', 'Semester ')
    df['Course_Code'] = df['STUDENT ID'].str.extract(r'^(\d{4})-')[0]
    
    # Determine course type
    df['Course_Type'] = df['Course_Code'].apply(
        lambda x: 'Certificate' if int(x) < 2000 else ('Diploma' if int(x) < 3000 else 'Specialist')
    )
    
    return df
# Load the data immediately so it's available for the app
df = load_and_prepare_data()
# Optional: Check if data loaded correctly
print(f"Data Loaded: {df.shape}")

Data Loaded: (505, 34)


In [397]:
# ============================================================================
# UPGRADED CHART 1: COURSE DIFFICULTY BUBBLE with CROSS-FILTERING
# ============================================================================

def create_interactive_course_bubble(df, selected_courses=None):
    """
    UPGRADED: Interactive bubble chart showing course difficulty
    Click any bubble to filter the entire dashboard by that course
    Shows GPA vs Failure Rate with enrollment as bubble size
    """
    
    # Calculate course-level statistics
    course_stats = df.groupby('Course_Code').agg({
        'GPA': 'mean',
        'Pass_Status': lambda x: (x == 'Fail').sum() / len(x) * 100,
        'STUDENT ID': 'nunique'
    }).reset_index()
    
    course_stats.columns = ['Course_Code', 'Avg_GPA', 'Failure_Rate', 'Enrollment']
    
    # Filter if specific courses selected
    if selected_courses:
        course_stats = course_stats[course_stats['Course_Code'].isin(selected_courses)]
    
    # Create bubble chart
    fig = px.scatter(course_stats,
                     x='Avg_GPA',
                     y='Failure_Rate',
                     size='Enrollment',
                     color='Failure_Rate',
                     hover_name='Course_Code',
                     text='Course_Code',
                     hover_data={
                         'Avg_GPA': ':.2f',
                         'Failure_Rate': ':.1f',
                         'Enrollment': ':,',
                         'Course_Code': False
                     },
                     size_max=60,
                     color_continuous_scale=[
                         [0, COLORS['success']],
                         [0.3, COLORS['primary']],
                         [0.6, COLORS['warning']],
                         [1, COLORS['danger']]
                     ],
                     range_color=[0, 50])
    
    # Update text position and styling
    fig.update_traces(
        textposition='middle center',
        textfont=dict(size=10, color=COLORS['text_primary'], family='monospace'),
        marker=dict(
            line=dict(width=2, color=COLORS['border']),
            opacity=0.8
        )
    )
    
    layout_config = CHART_TEMPLATE['layout'].copy()
    layout_config.update({
        'title': {
            'text': '<b>📊 Course Difficulty Matrix</b><br><sub>Click any bubble to filter dashboard | Size=Enrollment | Color=Failure Rate</sub>',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 16, 'color': COLORS['text_primary']}
        },
        'xaxis_title': 'Average GPA',
        'yaxis_title': 'Failure Rate (%)',
        'showlegend': False,
        'height': 450
    })
    
    fig.update_layout(**layout_config)
    
    # Add quadrant lines for interpretation
    fig.add_hline(y=25, line_dash="dash", line_color=COLORS['border'], 
                  opacity=0.5, line_width=1)
    fig.add_vline(x=3.0, line_dash="dash", line_color=COLORS['border'], 
                  opacity=0.5, line_width=1)
    
    # Add quadrant annotations
    fig.add_annotation(x=3.5, y=45, text="✅ Low Risk<br>High GPA, Low Failure", 
                      showarrow=False, font=dict(color=COLORS['success'], size=11),
                      bgcolor=COLORS['surface'], bordercolor=COLORS['success'], 
                      borderwidth=1, borderpad=4, opacity=0.8)
    
    fig.add_annotation(x=2.5, y=45, text="🔴 HIGH RISK<br>Low GPA, High Failure", 
                      showarrow=False, font=dict(color=COLORS['danger'], size=11, weight='bold'),
                      bgcolor=COLORS['surface'], bordercolor=COLORS['danger'], 
                      borderwidth=2, borderpad=4, opacity=0.9)
    
    fig.add_annotation(x=3.5, y=5, text="⚠️ Moderate<br>Good GPA, Some Failures", 
                      showarrow=False, font=dict(color=COLORS['warning'], size=10),
                      bgcolor=COLORS['surface'], bordercolor=COLORS['warning'], 
                      borderwidth=1, borderpad=4, opacity=0.8)
    
    fig.add_annotation(x=2.5, y=5, text="📉 Needs Support<br>Low GPA, Variable Outcomes", 
                      showarrow=False, font=dict(color=COLORS['info'], size=10),
                      bgcolor=COLORS['surface'], bordercolor=COLORS['info'], 
                      borderwidth=1, borderpad=4, opacity=0.8)
    
    # Make bubbles clickable
    fig.update_traces(
        customdata=course_stats[['Course_Code']],
        hovertemplate='<b>Course: %{customdata[0]}</b><br>' +
                      'Avg GPA: %{x:.2f}<br>' +
                      'Failure Rate: %{y:.1f}%<br>' +
                      'Enrollment: %{marker.size} students<br>' +
                      '<i>Click to filter dashboard</i><extra></extra>'
    )
    
    return fig

# ============================================================================
# UPGRADED CHART 2: SMART TRAJECTORY (LINE or GAUGE)
# ============================================================================

def create_smart_trajectory(df, view_mode='risk_level', selected_filter=None):
    """
    UPGRADED: Intelligently switches between LINE CHART (diploma) and GAUGE (certificate)
    Handles edge cases gracefully
    """
    
    # Apply cross-filter if exists
    if selected_filter:
        df = df[df['Course_Code'].isin(selected_filter)]
    
    # Detect if filtered data contains only certificates (1 semester courses)
    semester_counts = df.groupby('STUDENT ID')['PERIOD'].nunique()
    
    # If average semesters < 2, it's mostly certificates → use GAUGE
    if semester_counts.mean() < 1.5:
        return create_gauge_chart(df)
    else:
        return create_line_trajectory(df, view_mode)


def create_line_trajectory(df, view_mode):
    """Traditional line chart for multi-semester courses"""
    
    # Filter to students with at least 2 semesters
    retained_students = df.groupby('STUDENT ID')['PERIOD'].nunique()
    retained_students = retained_students[retained_students >= 2].index
    df_retained = df[df['STUDENT ID'].isin(retained_students)].copy()
    
    # IMPORTANT: Remove Semester 4 (only 1 student) to avoid misleading data
    df_retained = df_retained[df_retained['PERIOD'] != 'Sem 4']
    
    # Also filter out age groups with very few students (< 5)
    age_counts = df_retained.groupby('Age_Group').size()
    valid_ages = age_counts[age_counts >= 5].index
    df_retained = df_retained[df_retained['Age_Group'].isin(valid_ages)]
    
    fig = go.Figure()
    traces = {}
    
    # By Risk Level
    for risk in ['High Risk', 'Medium Risk', 'Low Risk']:
        risk_data = df_retained[df_retained['Initial_Risk'] == risk]
        if len(risk_data) == 0:
            continue
        trajectory = risk_data.groupby('Period_Clean')['GPA'].mean().reset_index()
        
        color_map = {'High Risk': COLORS['danger'], 
                     'Medium Risk': COLORS['warning'], 
                     'Low Risk': COLORS['success']}
        
        traces[f'risk_{risk}'] = go.Scatter(
            x=trajectory['Period_Clean'],
            y=trajectory['GPA'],
            mode='lines+markers',
            name=risk,
            line=dict(color=color_map[risk], width=3),
            marker=dict(size=10, symbol='circle'),
            visible=(view_mode == 'risk_level'),
            hovertemplate='<b>%{x}</b><br>Avg GPA: %{y:.2f}<extra></extra>'
        )
    
    # By Age Group (only include groups with sufficient data)
    for age_grp in valid_ages:
        age_data = df_retained[df_retained['Age_Group'] == age_grp]
        if len(age_data) == 0:
            continue
        trajectory = age_data.groupby('Period_Clean')['GPA'].mean().reset_index()
        
        color_map = {'18-25': '#3b82f6', '26-35': '#8b5cf6', 
                     '36-45': '#ec4899', '46+': '#f97316'}
        
        traces[f'age_{age_grp}'] = go.Scatter(
            x=trajectory['Period_Clean'],
            y=trajectory['GPA'],
            mode='lines+markers',
            name=f'Age {age_grp}',
            line=dict(color=color_map.get(age_grp, '#3b82f6'), width=3),
            marker=dict(size=10),
            visible=(view_mode == 'age_group'),
            hovertemplate='<b>%{x}</b><br>Avg GPA: %{y:.2f}<extra></extra>'
        )
    
    # All Students
    all_trajectory = df_retained.groupby('Period_Clean')['GPA'].mean().reset_index()
    traces['all_students'] = go.Scatter(
        x=all_trajectory['Period_Clean'],
        y=all_trajectory['GPA'],
        mode='lines+markers',
        name='All Students',
        line=dict(color=COLORS['primary'], width=4),
        marker=dict(size=12, symbol='diamond'),
        visible=(view_mode == 'all'),
        hovertemplate='<b>%{x}</b><br>Avg GPA: %{y:.2f}<extra></extra>'
    )
    
    for trace in traces.values():
        fig.add_trace(trace)
    
    dropdown_buttons = [
        dict(label='📊 By Initial Risk Level',
             method='update',
             args=[{'visible': [k.startswith('risk_') for k in traces.keys()]},
                   {'title.text': '<b>GPA Trajectory by Risk Level</b><br><sub>Students who completed 2-3 semesters (Sem 4 excluded)</sub>'}]),
        dict(label='👥 By Age Group',
             method='update',
             args=[{'visible': [k.startswith('age_') for k in traces.keys()]},
                   {'title.text': '<b>GPA Trajectory by Age</b><br><sub>Age groups with sufficient data only</sub>'}]),
        dict(label='🌐 All Students',
             method='update',
             args=[{'visible': [k == 'all_students' for k in traces.keys()]},
                   {'title.text': '<b>Overall GPA Trajectory</b><br><sub>Average across all students (Sem 1-3)</sub>'}])
    ]
    
    layout_config = CHART_TEMPLATE['layout'].copy()
    layout_config.update({
        'title': {
            'text': '<b>GPA Trajectory by Risk Level</b><br><sub>Students who completed 2-3 semesters (Sem 4 excluded)</sub>',
        },
        'xaxis_title': 'Semester',
        'yaxis_title': 'Average GPA',
        'yaxis_range': [1.5, 4.0],
        'hovermode': 'x unified',
        'height': 450,
        'updatemenus': [{
            'buttons': dropdown_buttons,
            'direction': "down",
            'pad': {"r": 10, "t": 10},
            'showactive': True,
            'x': 1.25,
            'xanchor': "right",
            'y': 1.28,
            'yanchor': "top",
            'bgcolor': COLORS['card'],
            'bordercolor': COLORS['primary'],
            'borderwidth': 2,
            'font': dict(color=COLORS['text_primary'], size=11)
        }],
        'margin': {'l': 60, 'r': 40, 't': 120, 'b': 60}
    })
    
    fig.update_layout(**layout_config)
    fig.add_hline(y=2.0, line_dash="dash", line_color=COLORS['danger'],
                  annotation_text="Passing Threshold (2.0)", annotation_position="right")
    
    return fig


def create_gauge_chart(df):
    """GAUGE chart for single-semester (Certificate) courses"""
    
    avg_gpa = df['GPA'].mean()
    pass_rate = (df['Pass_Status'] == 'Pass').sum() / len(df) * 100
    
    fig = go.Figure()
    
    # Main GPA Gauge
    fig.add_trace(go.Indicator(
        mode="gauge+number+delta",
        value=avg_gpa,
        domain={'x': [0, 0.48], 'y': [0.2, 0.8]},
        title={'text': "<b>Average GPA</b>", 'font': {'size': 16, 'color': COLORS['text_primary']}},
        delta={'reference': 2.5, 'increasing': {'color': COLORS['success']}, 'decreasing': {'color': COLORS['danger']}},
        number={'font': {'size': 40, 'color': COLORS['primary']}},
        gauge={
            'axis': {'range': [0, 4], 'tickwidth': 2, 'tickcolor': COLORS['text_secondary']},
            'bar': {'color': COLORS['primary']},
            'bgcolor': COLORS['surface'],
            'borderwidth': 2,
            'bordercolor': COLORS['border'],
            'steps': [
                {'range': [0, 2.0], 'color': COLORS['danger']},
                {'range': [2.0, 2.5], 'color': COLORS['warning']},
                {'range': [2.5, 4.0], 'color': COLORS['success']}
            ],
            'threshold': {
                'line': {'color': "white", 'width': 4},
                'thickness': 0.75,
                'value': avg_gpa
            }
        }
    ))
    
    # Pass Rate Gauge
    fig.add_trace(go.Indicator(
        mode="gauge+number",
        value=pass_rate,
        domain={'x': [0.52, 1], 'y': [0.2, 0.8]},
        title={'text': "<b>Pass Rate</b>", 'font': {'size': 16, 'color': COLORS['text_primary']}},
        number={'suffix': "%", 'font': {'size': 40, 'color': COLORS['success']}},
        gauge={
            'axis': {'range': [0, 100], 'tickwidth': 2, 'tickcolor': COLORS['text_secondary']},
            'bar': {'color': COLORS['success']},
            'bgcolor': COLORS['surface'],
            'borderwidth': 2,
            'bordercolor': COLORS['border'],
            'steps': [
                {'range': [0, 60], 'color': COLORS['danger']},
                {'range': [60, 80], 'color': COLORS['warning']},
                {'range': [80, 100], 'color': COLORS['success']}
            ]
        }
    ))
    
    layout_config = CHART_TEMPLATE['layout'].copy()
    layout_config.update({
        'title': {
            'text': '<b>📊 Certificate Course Performance Snapshot</b><br><sub>Single semester - showing gauges instead of trajectory</sub>',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 16, 'color': COLORS['text_primary']}
        },
        'height': 400
    })
    
    fig.update_layout(**layout_config)
    
    return fig

# ============================================================================
# UPGRADED CHART 3: MULTI-METRIC HEATMAP with RADIO BUTTONS
# ============================================================================

def create_multi_metric_heatmap(df, metric='failure_rate', gpa_threshold=2.0, selected_filter=None):
    """
    UPGRADED: Toggle between 3 metrics with DYNAMIC GPA THRESHOLD
    Now accepts a gpa_threshold parameter for flexible "failure" definition
    """
    
    if selected_filter:
        df = df[df['Course_Code'].isin(selected_filter)]
    
    # Filter out age groups with very few students
    age_counts = df.groupby('Age_Group').size()
    valid_ages = age_counts[age_counts >= 5].index
    df = df[df['Age_Group'].isin(valid_ages)]
    
    fig = go.Figure()
    metrics_data = {}
    
    # Metric 1: Failure Rate % (now uses dynamic threshold)
    failure_pivot = df.groupby(['Age_Group', 'Course_Code']).apply(
        lambda x: (x['GPA'] < gpa_threshold).sum() / len(x) * 100 if len(x) > 0 else 0
    ).reset_index()
    failure_pivot.columns = ['Age_Group', 'Course_Code', 'Value']
    failure_matrix = failure_pivot.pivot(index='Age_Group', columns='Course_Code', values='Value').fillna(0)
    
    hover_text_1 = []
    for i, age in enumerate(failure_matrix.index):
        row = []
        for j, course in enumerate(failure_matrix.columns):
            value = failure_matrix.iloc[i, j]
            count = len(df[(df['Age_Group'] == age) & (df['Course_Code'] == course)])
            text = f"<b>Age: {age}</b><br>Course: {course}<br>Below {gpa_threshold} GPA: {value:.1f}%<br>Students: {count}"
            row.append(text)
        hover_text_1.append(row)
    
    metrics_data['failure_rate'] = {
        'z': failure_matrix.values,
        'x': failure_matrix.columns.tolist(),
        'y': failure_matrix.index.tolist(),
        'colorscale': [[0, COLORS['success']], [0.3, COLORS['warning']], [1, COLORS['danger']]],
        'text': hover_text_1,
        'title': f'<b>🔴 Risk Hotspot: % Below {gpa_threshold} GPA</b><br><sub>Percentage of students below threshold by age & course</sub>',
        'colorbar_title': f'% < {gpa_threshold}'
    }
    
    # Metric 2: Average GPA
    gpa_pivot = df.groupby(['Age_Group', 'Course_Code'])['GPA'].mean().reset_index()
    gpa_pivot.columns = ['Age_Group', 'Course_Code', 'Value']
    gpa_matrix = gpa_pivot.pivot(index='Age_Group', columns='Course_Code', values='Value').fillna(0)
    
    hover_text_2 = []
    for i, age in enumerate(gpa_matrix.index):
        row = []
        for j, course in enumerate(gpa_matrix.columns):
            value = gpa_matrix.iloc[i, j]
            count = len(df[(df['Age_Group'] == age) & (df['Course_Code'] == course)])
            text = f"<b>Age: {age}</b><br>Course: {course}<br>Avg GPA: {value:.2f}<br>Students: {count}"
            row.append(text)
        hover_text_2.append(row)
    
    metrics_data['avg_gpa'] = {
        'z': gpa_matrix.values,
        'x': gpa_matrix.columns.tolist(),
        'y': gpa_matrix.index.tolist(),
        'colorscale': [[0, COLORS['danger']], [0.5, COLORS['warning']], [1, COLORS['success']]],
        'text': hover_text_2,
        'title': '<b>📊 Performance Heatmap: Average GPA</b><br><sub>Performance levels by age and course</sub>',
        'colorbar_title': 'Avg GPA'
    }
    
    # Metric 3: Average Attendance Rate
    attendance_pivot = df.groupby(['Age_Group', 'Course_Code'])['ATTENDANCE'].mean().reset_index()
    attendance_pivot.columns = ['Age_Group', 'Course_Code', 'Value']
    attendance_matrix = attendance_pivot.pivot(index='Age_Group', columns='Course_Code', values='Value').fillna(0)
    
    hover_text_3 = []
    for i, age in enumerate(attendance_matrix.index):
        row = []
        for j, course in enumerate(attendance_matrix.columns):
            value = attendance_matrix.iloc[i, j]
            count = len(df[(df['Age_Group'] == age) & (df['Course_Code'] == course)])
            text = f"<b>Age: {age}</b><br>Course: {course}<br>Avg Attendance: {value:.1f}%<br>Students: {count}"
            row.append(text)
        hover_text_3.append(row)
    
    metrics_data['attendance'] = {
        'z': attendance_matrix.values,
        'x': attendance_matrix.columns.tolist(),
        'y': attendance_matrix.index.tolist(),
        'colorscale': [[0, COLORS['danger']], [0.5, COLORS['warning']], [1, COLORS['success']]],
        'text': hover_text_3,
        'title': '<b>📅 Discipline Heatmap: Attendance Rates</b><br><sub>Average attendance by age and course</sub>',
        'colorbar_title': 'Attendance %'
    }
    
    # Create heatmap traces
    for metric_key, metric_info in metrics_data.items():
        fig.add_trace(go.Heatmap(
            z=metric_info['z'],
            x=metric_info['x'],
            y=metric_info['y'],
            colorscale=metric_info['colorscale'],
            text=metric_info['text'],
            hovertemplate='%{text}<extra></extra>',
            showscale=True,
            colorbar=dict(
                title=metric_info['colorbar_title'],
                tickfont=dict(color=COLORS['text_secondary']),
                outlinecolor=COLORS['border']
            ),
            visible=(metric == metric_key)
        ))
    
    # Radio buttons
    radio_buttons = [
        dict(label=f'🔴 Below {gpa_threshold} GPA %',
             method='update',
             args=[{'visible': [True, False, False]},
                   {'title.text': metrics_data['failure_rate']['title']}]),
        dict(label='📊 Average GPA',
             method='update',
             args=[{'visible': [False, True, False]},
                   {'title.text': metrics_data['avg_gpa']['title']}]),
        dict(label='📅 Attendance Rate',
             method='update',
             args=[{'visible': [False, False, True]},
                   {'title.text': metrics_data['attendance']['title']}])
    ]
    
    layout_config = CHART_TEMPLATE['layout'].copy()
    layout_config.update({
        'title': metrics_data[metric]['title'],
        'xaxis_title': 'Course Code',
        'yaxis_title': 'Age Group',
        'height': 500,
        'updatemenus': [{
            'buttons': radio_buttons,
            'direction': "down",
            'pad': {"r": 10, "t": 10},
            'showactive': True,
            'active': 0,
            'x': 1.0,
            'xanchor': "right",
            'y': 1.28,
            'yanchor': "top",
            'bgcolor': COLORS['card'],
            'bordercolor': COLORS['primary'],
            'borderwidth': 2,
            'font': dict(color=COLORS['text_primary'], size=11)
        }],
        'margin': {'l': 60, 'r': 40, 't': 130, 'b': 60}
    })
    
    fig.update_layout(**layout_config)
    
    return fig

# ============================================================================
# CHART 4: ATTENDANCE THRESHOLD (Enhanced)
# ============================================================================

def create_attendance_threshold_slider(df, threshold=75, selected_filter=None):
    """Enhanced attendance threshold analysis with proper spacing"""
    
    if selected_filter:
        df = df[df['Course_Code'].isin(selected_filter)]
    
    attendance_bands = list(range(50, 101, 5))
    threshold_stats = []
    
    for band in attendance_bands:
        band_data = df[df['ATTENDANCE'] >= band]
        if len(band_data) > 0:
            pass_rate = (band_data['Pass_Status'] == 'Pass').sum() / len(band_data) * 100
            avg_gpa = band_data['GPA'].mean()
            student_count = len(band_data)
            at_risk = (band_data['GPA'] < 2.5).sum()
        else:
            pass_rate = avg_gpa = student_count = at_risk = 0
        
        threshold_stats.append({
            'Threshold': f'{band}%+',
            'Band': band,
            'Pass_Rate': pass_rate,
            'Avg_GPA': avg_gpa,
            'Student_Count': student_count,
            'At_Risk': at_risk
        })
    
    stats_df = pd.DataFrame(threshold_stats)
    
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    
    fig.add_trace(
        go.Bar(
            x=stats_df['Threshold'],
            y=stats_df['Pass_Rate'],
            name='Pass Rate %',
            marker_color=COLORS['primary'],
            text=stats_df['Pass_Rate'].round(1),
            texttemplate='%{text}%',
            textposition='outside',
            textfont=dict(size=10),  # Smaller text to prevent overlap
            hovertemplate='<b>Attendance: %{x}</b><br>Pass Rate: %{y:.1f}%<extra></extra>'
        ),
        secondary_y=False
    )
    
    fig.add_trace(
        go.Scatter(
            x=stats_df['Threshold'],
            y=stats_df['Avg_GPA'],
            name='Average GPA',
            mode='lines+markers',
            line=dict(color=COLORS['danger'], width=3),
            marker=dict(size=10, symbol='diamond'),
            yaxis='y2',
            hovertemplate='<b>Attendance: %{x}</b><br>Avg GPA: %{y:.2f}<extra></extra>'
        ),
        secondary_y=True
    )
    
    threshold_idx = (threshold - 50) // 5
    if threshold_idx < len(stats_df):
        fig.add_vline(
            x=threshold_idx,
            line_dash="dash",
            line_color=COLORS['info'],
            line_width=2,
            annotation_text=f"Current: {threshold}%",
            annotation_position="top"
        )
    
    layout_config = CHART_TEMPLATE['layout'].copy()
    layout_config.update({
        'title': '<b>Attendance Threshold Impact</b><br><sub>Pass rate & GPA by minimum attendance</sub>',
        'xaxis_title': 'Minimum Attendance Requirement',
        'height': 500,  # Significantly increased height
        'hovermode': 'x unified',
        'legend': dict(
            orientation="h",
            yanchor="bottom",
            y=1.05,
            xanchor="right",
            x=1,
            bgcolor=COLORS['card'],
            bordercolor=COLORS['border'],
            borderwidth=1
        ),
        'margin': {'l': 60, 'r': 40, 't': 80, 'b': 80}  # More breathing room
    })
    
    fig.update_layout(**layout_config)
    fig.update_yaxes(title_text="Pass Rate (%)", secondary_y=False, range=[0, 110])  # Extended range for text labels
    fig.update_yaxes(title_text="Average GPA", secondary_y=True, range=[0, 4])
    
    return fig, stats_df

# ============================================================================
# UPGRADED KPI CARDS with SPARKLINES
# ============================================================================

def create_kpi_card_with_sparkline(title, value, subtitle, icon, sparkline_data=None, color='primary'):
    """
    UPGRADED KPI Card with mini sparkline trend chart
    """
    
    color_map = {
        'primary': COLORS['primary'],
        'danger': COLORS['danger'],
        'success': COLORS['success'],
        'info': COLORS['info'],
        'warning': COLORS['warning']  # ADDED WARNING COLOR
    }
    
    # Create sparkline if data provided
    sparkline_fig = None
    if sparkline_data is not None and len(sparkline_data) > 1:
        sparkline_fig = go.Figure()
        sparkline_fig.add_trace(go.Scatter(
            y=sparkline_data,
            mode='lines',
            line=dict(color=color_map[color], width=2),
            fill='tozeroy',
            fillcolor=f'rgba({int(color_map[color][1:3], 16)}, {int(color_map[color][3:5], 16)}, {int(color_map[color][5:7], 16)}, 0.2)',
            hovertemplate='Value: %{y:.1f}<extra></extra>'
        ))
        sparkline_fig.update_layout(
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)',
            margin=dict(l=0, r=0, t=0, b=0),
            height=60,
            showlegend=False,
            xaxis=dict(visible=False),
            yaxis=dict(visible=False)
        )
        sparkline_fig.update_xaxes(showgrid=False, zeroline=False)
        sparkline_fig.update_yaxes(showgrid=False, zeroline=False)
    
    card_content = [
        html.Div([
            html.Span(icon, style={
                'fontSize': '2rem',
                'color': color_map[color],
                'marginRight': '10px'
            }),
            html.Div([
                html.H6(title, style={
                    'color': COLORS['text_secondary'],
                    'fontSize': '0.875rem',
                    'fontWeight': '500',
                    'marginBottom': '0.25rem'
                }),
                html.H3(value, style={
                    'color': COLORS['text_primary'],
                    'fontSize': '1.875rem',
                    'fontWeight': '700',
                    'marginBottom': '0.25rem'
                }),
                html.P(subtitle, style={
                    'color': COLORS['text_secondary'],
                    'fontSize': '0.75rem',
                    'marginBottom': '0'
                })
            ])
        ], style={'display': 'flex', 'alignItems': 'center'})
    ]
    
    # Add sparkline if available
    if sparkline_fig:
        card_content.append(
            dcc.Graph(
                figure=sparkline_fig,
                config={'displayModeBar': False},
                style={'marginTop': '0.5rem'}
            )
        )
    
    return dbc.Card([
        dbc.CardBody(card_content)
    ], style={
        'backgroundColor': COLORS['card'],
        'border': f'1px solid {COLORS["border"]}',
        'borderRadius': '8px',
        'marginBottom': '1rem'
    })

# ============================================================================
# KPI CALCULATION
# ============================================================================

def calculate_kpis(df, selected_period=None, selected_course=None, selected_filter=None):
    """Calculate KPIs with historical data for sparklines"""
    
    filtered_df = df.copy()
    
    if selected_filter:
        filtered_df = filtered_df[filtered_df['Course_Code'].isin(selected_filter)]
    
    if selected_period and selected_period != 'all':
        filtered_df = filtered_df[filtered_df['PERIOD'] == selected_period]
    
    if selected_course and selected_course != 'all':
        filtered_df = filtered_df[filtered_df['Course_Code'] == selected_course]
    
    total_students = filtered_df['STUDENT ID'].nunique()
    at_risk_count = filtered_df[filtered_df['GPA'] < 2.5]['STUDENT ID'].nunique()
    at_risk_pct = (at_risk_count / total_students * 100) if total_students > 0 else 0
    avg_gpa = filtered_df['GPA'].mean()
    pass_rate = (filtered_df['Pass_Status'] == 'Pass').sum() / len(filtered_df) * 100 if len(filtered_df) > 0 else 0
    
    # Calculate sparkline data (GPA trend across semesters)
    gpa_sparkline = df.groupby('PERIOD')['GPA'].mean().values.tolist()
    
    # Calculate trend
    if selected_period and selected_period != 'all' and selected_period != 'Sem 1':
        prev_period = f'Sem {int(selected_period.split()[1]) - 1}'
        prev_gpa = df[df['PERIOD'] == prev_period]['GPA'].mean()
        gpa_trend = 'up' if avg_gpa > prev_gpa else 'down'
        gpa_change = abs(avg_gpa - prev_gpa)
    else:
        gpa_trend = 'neutral'
        gpa_change = 0
    
    return {
        'total_students': total_students,
        'at_risk_count': at_risk_count,
        'at_risk_pct': at_risk_pct,
        'avg_gpa': avg_gpa,
        'gpa_trend': gpa_trend,
        'gpa_change': gpa_change,
        'pass_rate': pass_rate,
        'gpa_sparkline': gpa_sparkline
    }

In [398]:
# Initialize the app
app = Dash(__name__, external_stylesheets=[dbc.themes.BOOTSTRAP])

# ... Copy the 'app.index_string' part if you want the custom CSS ...
# Inject custom CSS to fix dropdown readability
app.index_string = '''
<!DOCTYPE html>
<html>
    <head>
        {%metas%}
        <title>{%title%}</title>
        {%favicon%}
        {%css%}
        <style>
            /* Fix Plotly dropdown menu text visibility */
            .updatemenu-button {
                background-color: #1e3a5f !important;
                color: #f1f5f9 !important;
            }
            .updatemenu-button:hover {
                background-color: #2d4a6f !important;
                color: #fbbf24 !important;
            }
            .updatemenu-button.active {
                background-color: #fbbf24 !important;
                color: #0a1929 !important;
            }
            .updatemenu-item {
                background-color: #1e3a5f !important;
                color: #f1f5f9 !important;
            }
            .updatemenu-item:hover {
                background-color: #2d4a6f !important;
                color: #fbbf24 !important;
            }
            .updatemenu {
                background-color: #1e3a5f !important;
            }
        </style>
    </head>
    <body>
        {%app_entry%}
        <footer>
            {%config%}
            {%scripts%}
            {%renderer%}
        </footer>
    </body>
</html>
'''
# ... Copy the 'app.layout = ...' block here ...
app.layout = dbc.Container([
    
    # Store for cross-filter state
    dcc.Store(id='selected-courses-store', data=[]),
    
    # Header
    dbc.Row([
        dbc.Col([
            html.Div([
                html.H2([
                    html.Span("📊 ", style={'marginRight': '10px'}),
                    "Student Risk & Performance Monitor",
                ], style={'color': COLORS['text_primary'], 'fontWeight': '700', 'marginBottom': '0.5rem'}),
                html.P("Real-time analytics with cross-filtering, smart charts & enhanced interactivity", 
                       style={'color': COLORS['text_secondary'], 'fontSize': '0.95rem'})
            ], style={'padding': '1.5rem 0'})
        ])
    ]),
    
    # Dashboard Switcher
    dbc.Row([
        dbc.Col([
            dbc.ButtonGroup([
                dbc.Button("🎯 Thomas - Risk Monitor", id='btn-thomas', color='warning', className='active',
                          style={'backgroundColor': COLORS['primary'], 'borderColor': COLORS['primary'], 
                                'color': COLORS['background'], 'fontWeight': '600'}),
                dbc.Button("🤝 LinKai - Support Systems", id='btn-lingger', color='secondary', outline=True,
                          style={'borderColor': COLORS['border'], 'color': COLORS['text_primary']},
                          href='http://127.0.0.1:8051', external_link=True)
            ], style={'marginBottom': '1rem'})
        ], width=12)
    ]),
    
    # Global Filters
    dbc.Row([
        dbc.Col([
            html.Label("Select Semester", style={'color': COLORS['text_secondary'], 'fontSize': '0.875rem'}),
            dcc.Dropdown(id='semester-filter',
                        options=[{'label': 'All Semesters', 'value': 'all'}] + 
                                [{'label': period, 'value': period} for period in sorted([p for p in df['PERIOD'].unique() if pd.notna(p)])],
                        value='all', clearable=False)
        ], width=2),
        
        dbc.Col([
            html.Label("Select Course", style={'color': COLORS['text_secondary'], 'fontSize': '0.875rem'}),
            dcc.Dropdown(id='course-filter',
                        options=[{'label': 'All Courses', 'value': 'all'}] +
                                [{'label': f'Course {code}', 'value': code} for code in sorted([c for c in df['Course_Code'].unique() if pd.notna(c)])],
                        value='all', clearable=False)
        ], width=2),
        
        dbc.Col([
            html.Label("Risk Level Filter", style={'color': COLORS['text_secondary'], 'fontSize': '0.875rem'}),
            dcc.Dropdown(id='risk-filter',
                        options=[{'label': 'All Risk Levels', 'value': 'all'},
                                {'label': '🔴 High Risk Only', 'value': 'High Risk'},
                                {'label': '🟡 Medium Risk Only', 'value': 'Medium Risk'},
                                {'label': '🟢 Low Risk Only', 'value': 'Low Risk'}],
                        value='all', clearable=False)
        ], width=2),
        
        dbc.Col([
            html.Label("GPA Threshold", style={'color': COLORS['text_secondary'], 'fontSize': '0.875rem'}),
            dcc.Slider(id='gpa-threshold-slider', min=1.5, max=3.5, step=0.1, value=2.5,
                      marks={1.5: '1.5', 2.0: '2.0', 2.5: '2.5', 3.0: '3.0', 3.5: '3.5'},
                      tooltip={"placement": "bottom", "always_visible": True})
        ], width=3),
        
        dbc.Col([
            html.Label("Attendance Threshold", style={'color': COLORS['text_secondary'], 'fontSize': '0.875rem'}),
            dcc.Slider(id='global-attendance-slider', min=50, max=100, step=5, value=75,
                      marks={i: f'{i}%' for i in range(50, 101, 10)},
                      tooltip={"placement": "bottom", "always_visible": True})
        ], width=3)
    ], style={'marginBottom': '2rem'}),
    
    # Clear Filter Button
    dbc.Row([
        dbc.Col([
            dbc.Button("🔄 Clear Course Filter", id='clear-filter-btn', color='secondary', size='sm',
                      style={'backgroundColor': COLORS['card'], 'borderColor': COLORS['border'], 
                            'color': COLORS['text_primary']})
        ], width=12)
    ], style={'marginBottom': '1rem'}),
    
    # KPI Cards
    dbc.Row([
        dbc.Col(html.Div(id='kpi-total-students'), width=3),
        dbc.Col(html.Div(id='kpi-at-risk'), width=3),
        dbc.Col(html.Div(id='kpi-avg-gpa'), width=3),
        dbc.Col(html.Div(id='kpi-pass-rate'), width=3)
    ], style={'marginBottom': '2rem'}),
    
    # Chart 1: Course Difficulty Bubble (Full Width)
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id='chart-bubble', config={'displayModeBar': False})
                ])
            ], style={'backgroundColor': COLORS['card'], 'border': f'1px solid {COLORS["border"]}', 'borderRadius': '8px'})
        ], width=12)
    ], style={'marginBottom': '1.5rem'}),
    
    # Charts 2 & 3 (Side by Side)
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id='chart-trajectory', config={'displayModeBar': False})
                ])
            ], style={'backgroundColor': COLORS['card'], 'border': f'1px solid {COLORS["border"]}', 'borderRadius': '8px'})
        ], width=6),
        
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id='chart-heatmap', config={'displayModeBar': False})
                ])
            ], style={'backgroundColor': COLORS['card'], 'border': f'1px solid {COLORS["border"]}', 'borderRadius': '8px'})
        ], width=6)
    ], style={'marginBottom': '1.5rem'}),
    
    # Chart 4 (Full Width)
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id='chart-attendance', config={'displayModeBar': False}),
                    html.Div(id='attendance-insights', style={'marginTop': '1rem'})
                ])
            ], style={'backgroundColor': COLORS['card'], 'border': f'1px solid {COLORS["border"]}', 'borderRadius': '8px'})
        ], width=12)
    ], style={'marginBottom': '2rem'}),
    
    # Footer
    dbc.Row([
        dbc.Col([
            html.Hr(style={'borderColor': COLORS['border']}),
            html.P([
                "CA2 Data Visualization Assignment • ST1502 • ",
                html.Span("Thomas's Dashboard []", style={'color': COLORS['primary'], 'fontWeight': '600'}),
                " • AY2526 Sem 2 • ",
                html.Span("✨ With Cross-Filtering, Smart Charts & Sparklines", style={'color': COLORS['success'], 'fontSize': '0.8rem'})
            ], style={'textAlign': 'center', 'color': COLORS['text_secondary'], 'fontSize': '0.875rem', 'marginTop': '1rem'})
        ])
    ])
    
], fluid=True, style={'backgroundColor': COLORS['background'], 'minHeight': '100vh', 'padding': '2rem'})
# Ensure 'app.layout' is fully defined in this cell

In [399]:
# Callback for bubble chart click (cross-filtering)
@app.callback(
    Output('selected-courses-store', 'data'),
    Input('chart-bubble', 'clickData'),
    Input('clear-filter-btn', 'n_clicks'),
    prevent_initial_call=True
)
def update_cross_filter(clickData, clear_clicks):
    """Handle bubble chart clicks for cross-filtering"""
    ctx = dash.callback_context
    
    if not ctx.triggered:
        return []
    
    trigger_id = ctx.triggered[0]['prop_id'].split('.')[0]
    
    if trigger_id == 'clear-filter-btn':
        return []
    
    if clickData and 'points' in clickData:
        # Get the course code from customdata
        course_code = clickData['points'][0]['customdata'][0]
        return [course_code]
    
    return []


# Main dashboard update callback
@app.callback(
    [Output('kpi-total-students', 'children'),
     Output('kpi-at-risk', 'children'),
     Output('kpi-avg-gpa', 'children'),
     Output('kpi-pass-rate', 'children'),
     Output('chart-bubble', 'figure'),
     Output('chart-trajectory', 'figure'),
     Output('chart-heatmap', 'figure'),
     Output('chart-attendance', 'figure'),
     Output('attendance-insights', 'children')],
    [Input('semester-filter', 'value'),
     Input('course-filter', 'value'),
     Input('risk-filter', 'value'),
     Input('gpa-threshold-slider', 'value'),
     Input('global-attendance-slider', 'value'),
     Input('selected-courses-store', 'data')]
)
def update_dashboard(semester, course, risk_level, gpa_threshold, attendance_threshold, selected_courses):
    """Main callback with cross-filtering support and dynamic GPA threshold"""
    
    # Filter data
    filtered_df = df.copy()
    
    if semester != 'all':
        filtered_df = filtered_df[filtered_df['PERIOD'] == semester]
    
    if course != 'all':
        filtered_df = filtered_df[filtered_df['Course_Code'] == course]
    
    if risk_level != 'all':
        filtered_df = filtered_df[filtered_df['Initial_Risk'] == risk_level]
    
    # Calculate KPIs (using the dynamic GPA threshold)
    total_students = filtered_df['STUDENT ID'].nunique()
    at_risk_count = filtered_df[filtered_df['GPA'] < gpa_threshold]['STUDENT ID'].nunique()
    at_risk_pct = (at_risk_count / total_students * 100) if total_students > 0 else 0
    avg_gpa = filtered_df['GPA'].mean()
    pass_rate = (filtered_df['GPA'] >= 2.0).sum() / len(filtered_df) * 100 if len(filtered_df) > 0 else 0
    
    # Create KPI cards WITHOUT sparklines
    kpi1 = create_kpi_card_with_sparkline(
        "Total Students", f"{total_students:,}", "Unique students in dataset", "👥",
        sparkline_data=None, color='info'
    )
    
    kpi2 = create_kpi_card_with_sparkline(
        "At-Risk Students", f"{at_risk_count:,}",
        f"{at_risk_pct:.1f}% below {gpa_threshold} GPA threshold", "🔴",
        sparkline_data=None, color='danger'
    )
    
    kpi3 = create_kpi_card_with_sparkline(
        "Average GPA", f"{avg_gpa:.2f}", "Current selection", "📊",
        sparkline_data=None,
        color='success' if avg_gpa >= 3.0 else 'warning'
    )
    
    kpi4 = create_kpi_card_with_sparkline(
        "Pass Rate", f"{pass_rate:.1f}%", "Students with GPA ≥ 2.0", "✅",
        sparkline_data=None, color='success' if pass_rate >= 80 else 'danger'
    )
    
    # Generate charts
    chart1 = create_interactive_course_bubble(filtered_df, selected_courses if selected_courses else None)
    chart2 = create_smart_trajectory(filtered_df, selected_filter=selected_courses if selected_courses else None)
    chart3 = create_multi_metric_heatmap(filtered_df, gpa_threshold=gpa_threshold, 
                                         selected_filter=selected_courses if selected_courses else None)
    chart4, threshold_stats = create_attendance_threshold_slider(filtered_df, attendance_threshold,
                                                                  selected_filter=selected_courses if selected_courses else None)
    
    # Create attendance insights
    current_stats = threshold_stats[threshold_stats['Band'] == attendance_threshold].iloc[0]
    insights = dbc.Alert([
        html.H6("💡 Insights at Current Threshold:", style={'marginBottom': '0.5rem'}),
        html.Ul([
            html.Li(f"{current_stats['Student_Count']:,} students meet {attendance_threshold}% attendance requirement"),
            html.Li(f"{current_stats['Pass_Rate']:.1f}% pass rate among students at this threshold"),
            html.Li(f"Average GPA of {current_stats['Avg_GPA']:.2f} for students meeting requirement"),
            html.Li(f"{current_stats['At_Risk']:,} at-risk students (GPA < {gpa_threshold}) in this group")
        ], style={'marginBottom': 0})
    ], color='info', style={
        'backgroundColor': COLORS['surface'],
        'borderColor': COLORS['info'],
        'color': COLORS['text_primary']
    })
    
    return kpi1, kpi2, kpi3, kpi4, chart1, chart2, chart3, chart4, insights

In [400]:
# Run the app inside the notebook
# mode='inline' shows it in the output cell
# mode='external' opens it in a new browser tab (better for full-screen dashboards)
# mode='jupyterlab' opens it in a side pane (if using JupyterLab)

if __name__ == '__main__':
    app.run(jupyter_mode='external', port=8050)

Dash app running on http://127.0.0.1:8050/


# Lin Kai's dashboard setup

In [401]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import dash
from dash import dcc, html, Input, Output, State
import dash_bootstrap_components as dbc
from datetime import datetime
import numpy as np

# ============================================================================
# CONFIGURATION & STYLING
# ============================================================================

# Color scheme - Blue/Teal theme (complementary to Thomas's yellow/orange)
COLORS = {
    'background': '#0a1929',          # Dark navy blue (same as Thomas)
    'surface': '#132f4c',              # Slightly lighter navy
    'card': '#1e3a5f',                 # Card background
    'primary': '#06b6d4',              # Cyan/Teal accent (main difference)
    'secondary': '#0891b2',            # Darker teal  
    'success': '#10b981',              # Green
    'danger': '#ef4444',               # Red
    'warning': '#f59e0b',              # Orange
    'info': '#3b82f6',                 # Blue
    'text_primary': '#f1f5f9',         # Light text
    'text_secondary': '#94a3b8',       # Muted text
    'border': '#334155',               # Border color
    'grid': '#1e293b'                  # Grid lines
}

# Chart template configuration
CHART_TEMPLATE = {
    'layout': {
        'paper_bgcolor': COLORS['background'],
        'plot_bgcolor': COLORS['surface'],
        'font': {'color': COLORS['text_primary'], 'family': 'Inter, sans-serif'},
        'xaxis': {
            'gridcolor': COLORS['grid'],
            'linecolor': COLORS['border'],
            'tickfont': {'color': COLORS['text_secondary']}
        },
        'yaxis': {
            'gridcolor': COLORS['grid'],
            'linecolor': COLORS['border'],
            'tickfont': {'color': COLORS['text_secondary']}
        },
        'hovermode': 'closest',
        'margin': {'l': 60, 'r': 40, 't': 60, 'b': 60}
    }
}

In [402]:
def load_and_prepare_data():
    """Load and prepare the master dataset with all necessary calculations"""
    
    df = pd.read_csv('../cleaned_data/master_dataset.csv')
    
    # Drop rows with NaN in critical columns
    df = df.dropna(subset=['PERIOD', 'GPA', 'STUDENT ID'])
    
    # Data type conversions
    df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')
    df['COMMENCEMENT DATE'] = pd.to_datetime(df['COMMENCEMENT DATE'], errors='coerce')
    df['COMPLETION DATE'] = pd.to_datetime(df['COMPLETION DATE'], errors='coerce')
    
    # Create age groups
    df['Age_Group'] = pd.cut(df['AGE'], 
                              bins=[0, 35, 45, 100], 
                              labels=['18-35', '36-45', '46+'])
    
    # Risk classification
    df['Initial_Risk'] = pd.cut(df['GPA'].where(df['PERIOD'] == 'Sem 1'),
                                 bins=[0, 2.5, 3.0, 4.0],
                                 labels=['High Risk', 'Medium Risk', 'Low Risk'])
    df['Initial_Risk'] = df.groupby('STUDENT ID')['Initial_Risk'].transform('first')
    
    # Pass/Fail status
    df['Pass_Status'] = df['GPA'].apply(lambda x: 'Pass' if x >= 2.0 else 'Fail')
    
    # Handle missing values
    df['ATTENDANCE'] = df['ATTENDANCE'].fillna(0)
    df['SELF-STUDY HRS'] = df['SELF-STUDY HRS'].fillna(0)
    
    # Fill support columns with median
    support_cols = ['TEACHING SUPPORT', 'COMPANY SUPPORT', 'FAMILY SUPPORT', 'COURSE RELEVANCE']
    for col in support_cols:
        if col in df.columns:
            df[col] = df[col].fillna(df[col].median())
    
    # Attendance categories
    df['Attendance_Category'] = pd.cut(df['ATTENDANCE'],
                                        bins=[0, 60, 75, 85, 100],
                                        labels=['Critical (<60%)', 'Low (60-75%)', 
                                               'Good (75-85%)', 'Excellent (85%+)'])
    
    # Study hours categories
    df['Study_Category'] = pd.cut(df['SELF-STUDY HRS'],
                                   bins=[0, 5, 10, 15, 100],
                                   labels=['Minimal (0-5h)', 'Low (5-10h)', 
                                          'Moderate (10-15h)', 'High (15h+)'])
    
    # Clean period names
    df['Period_Clean'] = df['PERIOD'].str.replace('Sem ', 'Semester ')
    
    # Course code extraction
    df['Course_Code'] = df['STUDENT ID'].str.extract(r'^(\d{4})-')[0]
    
    return df


In [403]:
# ============================================================================
# CHART GENERATION FUNCTIONS
# ============================================================================

def create_nationality_study_effort(df, selected_nationality=None):
    """
    CHART 1: Nationality & Study Effort Comparison - Plotly Express Box Plot
    Shows study hours distribution by nationality status
    """
    
    # Filter if specific nationality selected
    if selected_nationality and selected_nationality != 'all':
        df_filtered = df[df['NATIONALITY_STATUS'] == selected_nationality]
    else:
        df_filtered = df.copy()
    
    # Create box plot
    fig = px.box(df_filtered,
                 x='NATIONALITY_STATUS',
                 y='SELF-STUDY HRS',
                 color='NATIONALITY_STATUS',
                 points='outliers',
                 hover_data=['GPA', 'ATTENDANCE'],
                 color_discrete_map={
                     'SG Citizen': '#3b82f6',
                     'SG PR': '#8b5cf6',
                     'Foreigner': '#06b6d4'
                 },
                 category_orders={'NATIONALITY_STATUS': ['SG Citizen', 'SG PR', 'Foreigner']})
    
    # Calculate averages for annotation
    avg_by_nat = df_filtered.groupby('NATIONALITY_STATUS')['SELF-STUDY HRS'].mean()
    
    layout_config = CHART_TEMPLATE['layout'].copy()
    layout_config.update({
        'title': {
            'text': '<b>Study Effort by Nationality Status</b><br><sub>Comparing self-study hours across student backgrounds</sub>',
            'x': 0.5,
            'xanchor': 'center',
            'font': {'size': 16, 'color': COLORS['text_primary']}
        },
        'xaxis_title': 'Nationality Status',
        'yaxis_title': 'Weekly Self-Study Hours',
        'showlegend': False,
        'height': 400
    })
    
    fig.update_layout(**layout_config)
    
    # --- ANNOTATIONS ---
    for nat, avg in avg_by_nat.items():
        fig.add_annotation(
            x=nat, 
            y=avg,
            text=f"<b>Avg: {avg:.1f}h</b>",  # HTML Bold tags
            showarrow=True,
            arrowhead=2,
            arrowsize=1,
            arrowwidth=2,
            arrowcolor=COLORS['primary'],    # Cyan Arrow
            ax=40,
            ay=-30,
            # --- STYLING BOX ---
            font=dict(size=13, color='#ffffff'), # Larger White Text
            bgcolor=COLORS['card'],              # Dark Card Background
            bordercolor=COLORS['primary'],       # Cyan Border
            borderwidth=2,                       # Thicker Border
            borderpad=5,                         # More breathing room
            opacity=0.95
        )
    
    return fig


def create_support_factors_impact(df, selected_factor='all'):
    """
    CHART 2: Support Factors - ALIGNED VERSION
    Fixes: Height=450 (matches Chart 3), Stretched to right, Filter aligned.
    """
    fig = go.Figure()
    
    # Support factors configuration
    support_factors = {
        'TEACHING SUPPORT': {'name': 'Teaching Support', 'color': '#3b82f6'},
        'COMPANY SUPPORT': {'name': 'Company Support', 'color': '#8b5cf6'},
        'FAMILY SUPPORT': {'name': 'Family Support', 'color': '#ec4899'},
        'COURSE RELEVANCE': {'name': 'Course Relevance', 'color': '#06b6d4'}
    }
    
    trace_keys = []
    
    # 1. Create Traces
    for col, info in support_factors.items():
        if col not in df.columns:
            continue
            
        support_impact = df.groupby(col)['GPA'].mean().reset_index().sort_values(col)
        trace_keys.append(col)
        
        fig.add_trace(go.Scatter(
            x=support_impact[col],
            y=support_impact['GPA'],
            mode='lines+markers',
            name=info['name'],
            line=dict(color=info['color'], width=3),
            marker=dict(size=12, symbol='circle'),
            visible=(selected_factor == 'all' or col == selected_factor),
            hovertemplate=f'<b>{info["name"]}</b><br>Level: %{{x}}<br>GPA: %{{y:.2f}}<extra></extra>'
        ))

    # 2. Dropdown Logic
    dropdown_buttons = [
        dict(
            label='📊 All Factors (Overlay)',
            method='update',
            args=[
                {'visible': [True] * len(trace_keys)},
                {'title.text': '<b>Support Factors Impact on GPA</b><br><sub>Comparing all environmental support systems</sub>'}
            ]
        )
    ]
    
    for i, (col, info) in enumerate(support_factors.items()):
        if col not in df.columns: continue
        
        visibility_list = [False] * len(trace_keys)
        if i < len(visibility_list):
            visibility_list[i] = True
            
        dropdown_buttons.append(dict(
            label=f"{info['name']}",
            method='update',
            args=[
                {'visible': visibility_list},
                {'title.text': f"<b>{info['name']} Impact on GPA</b><br><sub>Analysis of specific support factor</sub>"}
            ]
        ))

    # 3. Layout Configuration (UPDATED)
    layout_config = CHART_TEMPLATE['layout'].copy()
    layout_config.update({
        'title': {
            'text': '<b>Support Factors Impact on GPA</b><br><sub>Comparing all environmental support systems</sub>',
        },
        'xaxis_title': 'Support Level (1=Low, 5=High)',
        'yaxis_title': 'Average GPA',
        'yaxis_range': [1.5, 4.0],
        
        # --- ALIGNMENT FIXES ---
        'height': 450,                          # Increased to match Chart 3
        'margin': {'l': 60, 'r': 20, 't': 80, 'b': 60}, # r=20 stretches graph to right
        # -----------------------
        
        'updatemenus': [
            dict(
                buttons=dropdown_buttons,
                direction="down",
                pad={"r": 0, "t": 0},
                showactive=True,
                x=1.0,              # Top Right
                xanchor="right",
                y=1.15,
                yanchor="top",
                bgcolor=COLORS['card'],
                bordercolor=COLORS['primary'],
                borderwidth=1,
                font=dict(color=COLORS['text_primary'], size=11)
            )
        ]
    })
    
    fig.update_layout(**layout_config)
    
    # Add passing threshold line
    fig.add_hline(y=2.0, line_dash="dash", line_color=COLORS['danger'],
                  annotation_text="Passing Threshold", annotation_position="right")
    
    return fig

def create_attendance_study_compensation_heatmap(df, view_mode='pass_fail'):
    """
    CHART 3: Attendance vs Study Hours Matrix - ENHANCED
    Fixes: Stretched layout, numbers on all charts, clearer grid lines.
    """
    
    # Create bins for attendance and study hours
    df_analysis = df.copy()
    df_analysis['Att_Bin'] = pd.cut(df_analysis['ATTENDANCE'],
                                     bins=[0, 60, 75, 85, 100],
                                     labels=['<60%', '60-75%', '75-85%', '85%+'])
    df_analysis['Study_Bin'] = pd.cut(df_analysis['SELF-STUDY HRS'],
                                      bins=[0, 5, 10, 15, 100],
                                      labels=['0-5h', '5-10h', '10-15h', '15h+'])
    
    fig = go.Figure()
    
    # Common font settings for the numbers inside the boxes
    text_style = {"family": "Inter, sans-serif", "size": 14, "color": "white"}

    # --- View 1: Pass/Fail Zones ---
    passfail_matrix = df_analysis.groupby(['Study_Bin', 'Att_Bin']).apply(
        lambda x: (x['Pass_Status'] == 'Pass').sum() / len(x) * 100 if len(x) > 0 else 0
    ).reset_index()
    passfail_pivot = passfail_matrix.pivot(index='Study_Bin', columns='Att_Bin', values=0).fillna(0)
    
    hover_text_1 = []
    for i, study in enumerate(passfail_pivot.index):
        row = []
        for j, att in enumerate(passfail_pivot.columns):
            value = passfail_pivot.iloc[i, j]
            count = len(df_analysis[(df_analysis['Study_Bin'] == study) & (df_analysis['Att_Bin'] == att)])
            row.append(f"<b>Study: {study}</b><br>Attendance: {att}<br>Pass Rate: {value:.1f}%<br>Students: {count}")
        hover_text_1.append(row)
    
    fig.add_trace(go.Heatmap(
        z=passfail_pivot.values,
        x=passfail_pivot.columns.tolist(),
        y=passfail_pivot.index.tolist(),
        colorscale=[[0, COLORS['danger']], [0.5, COLORS['warning']], [1, COLORS['success']]],
        text=hover_text_1,
        texttemplate="%{z:.0f}%", # Adds numbers (e.g. 85%)
        textfont=text_style,
        xgap=2, ygap=2,           # Adds gaps for clarity
        hovertemplate='%{text}<extra></extra>',
        visible=(view_mode == 'pass_fail')
    ))
    
    # --- View 2: GPA Gradient ---
    gpa_matrix = df_analysis.groupby(['Study_Bin', 'Att_Bin'])['GPA'].mean().reset_index()
    gpa_pivot = gpa_matrix.pivot(index='Study_Bin', columns='Att_Bin', values='GPA').fillna(0)
    
    hover_text_2 = []
    for i, study in enumerate(gpa_pivot.index):
        row = []
        for j, att in enumerate(gpa_pivot.columns):
            value = gpa_pivot.iloc[i, j]
            count = len(df_analysis[(df_analysis['Study_Bin'] == study) & (df_analysis['Att_Bin'] == att)])
            row.append(f"<b>Study: {study}</b><br>Attendance: {att}<br>Avg GPA: {value:.2f}<br>Students: {count}")
        hover_text_2.append(row)
    
    fig.add_trace(go.Heatmap(
        z=gpa_pivot.values,
        x=gpa_pivot.columns.tolist(),
        y=gpa_pivot.index.tolist(),
        colorscale=[[0, '#1e293b'], [0.33, COLORS['danger']], [0.66, COLORS['warning']], [1, COLORS['success']]],
        text=hover_text_2,
        texttemplate="%{z:.2f}",  # Adds numbers (e.g. 3.42)
        textfont=text_style,
        xgap=2, ygap=2,
        hovertemplate='%{text}<extra></extra>',
        visible=(view_mode == 'gpa_gradient')
    ))
    
    # --- View 3: Student Count ---
    count_matrix = df_analysis.groupby(['Study_Bin', 'Att_Bin']).size().reset_index()
    count_pivot = count_matrix.pivot(index='Study_Bin', columns='Att_Bin', values=0).fillna(0)
    
    hover_text_3 = []
    for i, study in enumerate(count_pivot.index):
        row = []
        for j, att in enumerate(count_pivot.columns):
            value = int(count_pivot.iloc[i, j])
            row.append(f"<b>Study: {study}</b><br>Attendance: {att}<br>Students: {value}")
        hover_text_3.append(row)
    
    fig.add_trace(go.Heatmap(
        z=count_pivot.values,
        x=count_pivot.columns.tolist(),
        y=count_pivot.index.tolist(),
        colorscale=[[0, COLORS['surface']], [0.5, COLORS['info']], [1, COLORS['primary']]],
        text=hover_text_3,
        texttemplate="%{z:.0f}",  # Adds numbers (e.g. 120)
        textfont=text_style,
        xgap=2, ygap=2,
        hovertemplate='%{text}<extra></extra>',
        visible=(view_mode == 'student_count')
    ))
    
    # --- Radio Buttons (Fixed Location & Title Bug) ---
    radio_buttons = [
        dict(
            label='✅ Pass/Fail Zones',
            method='update',
            args=[{'visible': [True, False, False]},
                  {'title.text': '<b>Compensation Matrix: Pass/Fail</b><br><sub>Can high study hours fix low attendance?</sub>'}]
        ),
        dict(
            label='📊 GPA Gradient',
            method='update',
            args=[{'visible': [False, True, False]},
                  {'title.text': '<b>Performance Matrix: Avg GPA</b><br><sub>Average GPA across effort combinations</sub>'}]
        ),
        dict(
            label='👥 Student Distribution',
            method='update',
            args=[{'visible': [False, False, True]},
                  {'title.text': '<b>Population Matrix: Student Count</b><br><sub>Number of students in each effort category</sub>'}]
        )
    ]
    
    layout_config = CHART_TEMPLATE['layout'].copy()
    layout_config.update({
        'title': {
            'text': '<b>Compensation Matrix: Pass/Fail</b><br><sub>Can high study hours fix low attendance?</sub>',
        },
        'xaxis_title': 'Attendance Level',
        'yaxis_title': 'Weekly Study Hours',
        'height': 450,
        # Reduced margins to let the chart expand
        'margin': {'l': 60, 'r': 20, 't': 80, 'b': 60}, 
        'updatemenus': [
            dict(
                buttons=radio_buttons,
                direction="down",
                pad={"r": 0, "t": 0},
                showactive=True,
                x=1.0,              # Moves dropdown to inside-right (Fixed)
                xanchor="right",
                y=1.15,
                yanchor="top",
                bgcolor=COLORS['card'],
                bordercolor=COLORS['primary'],
                borderwidth=1,
                font=dict(color=COLORS['text_primary'], size=11)
            )
        ]
    })
    
    fig.update_layout(**layout_config)
    
    return fig

def create_age_attendance_discipline_slider(df, threshold=75):
    """
    CHART 4: Risk Composition - HORIZONTAL Stacked Bar
    Title Updated: More descriptive about the simulation aspect.
    """
    
    fig = go.Figure()
    
    age_groups = ['46+', '36-45', '18-35']
    
    risk_pcts = []
    safe_pcts = []
    risk_counts = []
    safe_counts = []
    risk_text = []
    safe_text = []
    
    for age_grp in age_groups:
        age_data = df[df['Age_Group'] == age_grp]
        total = len(age_data)
        
        at_risk = (age_data['ATTENDANCE'] < threshold).sum()
        safe = total - at_risk
        
        r_pct = (at_risk / total * 100) if total > 0 else 0
        s_pct = (safe / total * 100) if total > 0 else 0
        
        risk_pcts.append(r_pct)
        safe_pcts.append(s_pct)
        risk_counts.append(at_risk)
        safe_counts.append(safe)
        
        if r_pct > 5:
            risk_text.append(f"<b>{r_pct:.1f}%</b><br>({at_risk})")
        else:
            risk_text.append("")
            
        safe_text.append(f"<b>{s_pct:.1f}%</b>")
    
    # Trace 1: AT RISK
    fig.add_trace(go.Bar(
        name=f'Fails Requirement (< {threshold}%)', # Legend explains the logic
        y=age_groups,
        x=risk_pcts,
        orientation='h',
        marker_color=COLORS['danger'],
        text=risk_text,
        textposition='auto',
        hovertemplate='<b>Age: %{y}</b><br>Fail Rate: %{x:.1f}%<br>Students: %{customdata}<extra></extra>',
        customdata=risk_counts
    ))
    
    # Trace 2: SAFE
    fig.add_trace(go.Bar(
        name=f'Meets Requirement (≥ {threshold}%)', # Legend explains the logic
        y=age_groups,
        x=safe_pcts,
        orientation='h',
        marker_color=COLORS['success'],
        text=safe_text,
        textposition='auto',
        hovertemplate='<b>Age: %{y}</b><br>Pass Rate: %{x:.1f}%<br>Students: %{customdata}<extra></extra>',
        customdata=safe_counts
    ))

    layout_config = CHART_TEMPLATE['layout'].copy()
    layout_config.update({
        # --- NEW CLEARER TITLE ---
        'title': f'<b>Risk Scenarios: Who fails the attendance rule?</b><br><sub>Showing % of students below the {threshold}% threshold (Adjust slider below)</sub>',
        'xaxis_title': 'Percentage of Group',
        'yaxis_title': 'Age Group',
        'barmode': 'stack',
        'showlegend': True,
        'legend': {'orientation': 'h', 'yanchor': 'bottom', 'y': 1.02, 'xanchor': 'right', 'x': 1},
        'height': 350,
        'xaxis': {'range': [0, 100], 'gridcolor': COLORS['grid']},
        'margin': {'l': 80, 'r': 40, 't': 80, 'b': 40} 
    })
    
    fig.update_layout(**layout_config)
    
    # --- Stats return remains same ---
    summary_data = []
    normal_order = ['18-35', '36-45', '46+']
    for age in normal_order:
         age_data = df[df['Age_Group'] == age]
         total = len(age_data)
         at_risk = (age_data['ATTENDANCE'] < threshold).sum()
         summary_data.append({
            'Age_Group': age, 
            'At_Risk_Pct': (at_risk / total * 100) if total > 0 else 0,
            'At_Risk_Count': at_risk,
            'Total': total
         })
         
    return fig, pd.DataFrame(summary_data)

# ============================================================================
# KPI CALCULATION
# ============================================================================

def calculate_kpis(df, selected_period=None, selected_course=None):
    """Calculate KPIs based on filters"""
    
    filtered_df = df.copy()
    
    if selected_period and selected_period != 'all':
        filtered_df = filtered_df[filtered_df['PERIOD'] == selected_period]
    
    if selected_course and selected_course != 'all':
        filtered_df = filtered_df[filtered_df['Course_Code'] == selected_course]
    
    # Calculate metrics
    total_students = filtered_df['STUDENT ID'].nunique()
    
    # Support seeking percentage (students with any support > 3)
    if all(col in filtered_df.columns for col in ['TEACHING SUPPORT', 'COMPANY SUPPORT', 'FAMILY SUPPORT']):
        high_support = filtered_df[
            (filtered_df['TEACHING SUPPORT'] >= 4) |
            (filtered_df['COMPANY SUPPORT'] >= 4) |
            (filtered_df['FAMILY SUPPORT'] >= 4)
        ]['STUDENT ID'].nunique()
        support_seeking_pct = (high_support / total_students * 100) if total_students > 0 else 0
    else:
        support_seeking_pct = 0
    
    # Average support rating
    if all(col in filtered_df.columns for col in ['TEACHING SUPPORT', 'COMPANY SUPPORT', 'FAMILY SUPPORT']):
        avg_support = filtered_df[['TEACHING SUPPORT', 'COMPANY SUPPORT', 'FAMILY SUPPORT']].mean().mean()
    else:
        avg_support = 0
    
    # High risk percentage
    high_risk_count = filtered_df[filtered_df['GPA'] < 2.5]['STUDENT ID'].nunique()
    high_risk_pct = (high_risk_count / total_students * 100) if total_students > 0 else 0
    
    return {
        'total_students': total_students,
        'avg_support': avg_support,
        'high_risk_pct': high_risk_pct,
        'support_seeking_pct': support_seeking_pct
    }


def create_kpi_card(title, value, subtitle, icon, color='primary'):
    """Create a Bootstrap KPI card component"""
    
    color_map = {
        'primary': COLORS['primary'],
        'danger': COLORS['danger'],
        'success': COLORS['success'],
        'info': COLORS['info']
    }
    
    return dbc.Card([
        dbc.CardBody([
            html.Div([
                html.Span(icon, style={
                    'fontSize': '2rem',
                    'color': color_map[color],
                    'marginRight': '10px'
                }),
                html.Div([
                    html.H6(title, style={
                        'color': COLORS['text_secondary'],
                        'fontSize': '0.875rem',
                        'fontWeight': '500',
                        'marginBottom': '0.25rem'
                    }),
                    html.H3(value, style={
                        'color': COLORS['text_primary'],
                        'fontSize': '1.875rem',
                        'fontWeight': '700',
                        'marginBottom': '0.25rem'
                    }),
                    html.P(subtitle, style={
                        'color': COLORS['text_secondary'],
                        'fontSize': '0.75rem',
                        'marginBottom': '0'
                    })
                ])
            ], style={'display': 'flex', 'alignItems': 'center'})
        ])
    ], style={
        'backgroundColor': COLORS['card'],
        'border': f'1px solid {COLORS["border"]}',
        'borderRadius': '8px',
        'marginBottom': '1rem'
    })

In [404]:
# ============================================================================
# DASH APP INITIALIZATION
# ============================================================================

app = dash.Dash(
    __name__,
    external_stylesheets=[dbc.themes.BOOTSTRAP],
    suppress_callback_exceptions=True
)

# Load data
df = load_and_prepare_data()

# ============================================================================
# LAYOUT
# ============================================================================

app.layout = dbc.Container([
    
    # Header Section
    dbc.Row([
        dbc.Col([
            html.Div([
                html.H2([
                    html.Span("🤝 ", style={'marginRight': '10px'}),
                    "Student Support Ecosystem Dashboard"
                ], style={
                    'color': COLORS['text_primary'],
                    'fontWeight': '700',
                    'marginBottom': '0.5rem'
                }),
                html.P("Analyzing environmental factors and support systems that drive student success", 
                       style={'color': COLORS['text_secondary'], 'fontSize': '0.95rem'})
            ], style={'padding': '1.5rem 0'})
        ])
    ]),
    
    # Dashboard Switcher
    dbc.Row([
        dbc.Col([
            dbc.ButtonGroup([
                dbc.Button(
                    "🎯 Thomas - Risk Monitor",
                    id='btn-thomas',
                    color='secondary',
                    outline=True,
                    style={
                        'borderColor': COLORS['border'],
                        'color': COLORS['text_primary']
                    },
                    href='http://127.0.0.1:8050',
                    external_link=True
                ),
                dbc.Button(
                    "🤝 LinKai - Support Systems",
                    id='btn-lingger',
                    color='info',
                    className='active',
                    style={
                        'backgroundColor': COLORS['primary'],
                        'borderColor': COLORS['primary'],
                        'color': COLORS['background'],
                        'fontWeight': '600'
                    }
                )
            ], style={'marginBottom': '1rem'})
        ], width=12)
    ]),
    
    # Global Filters Row
    dbc.Row([
        # Filter 1: Semester
        dbc.Col([
            html.Label("Select Semester", style={'color': COLORS['text_secondary'], 'fontSize': '0.875rem'}),
            dcc.Dropdown(
                id='semester-filter',
                options=[{'label': 'All Semesters', 'value': 'all'}] + 
                        [{'label': period, 'value': period} for period in sorted([p for p in df['PERIOD'].unique() if pd.notna(p)])],
                value='all',
                clearable=False
            )
        ], width=3),
        
        # Filter 2: Course
        dbc.Col([
            html.Label("Select Course", style={'color': COLORS['text_secondary'], 'fontSize': '0.875rem'}),
            dcc.Dropdown(
                id='course-filter',
                options=[{'label': 'All Courses', 'value': 'all'}] +
                        [{'label': f'Course {code}', 'value': code} for code in sorted([c for c in df['Course_Code'].unique() if pd.notna(c)])],
                value='all',
                clearable=False
            )
        ], width=3),
        
        # Filter 3: Nationality (IGNORED BY CHART 1)
        dbc.Col([
            html.Label("Nationality Filter", style={'color': COLORS['text_secondary'], 'fontSize': '0.875rem'}),
            dcc.Dropdown(
                id='nationality-filter',
                options=[
                    {'label': 'All Nationalities', 'value': 'all'},
                    {'label': '🇸🇬 SG Citizen', 'value': 'SG Citizen'},
                    {'label': '🏠 SG PR', 'value': 'SG PR'},
                    {'label': '🌏 Foreigner', 'value': 'Foreigner'}
                ],
                value='all',
                clearable=False
            )
        ], width=3),
        
        # Filter 4: Age Group (NEW! REPLACES SLIDER) (IGNORED BY CHART 4)
        dbc.Col([
            html.Label("Age Group Filter", style={'color': COLORS['text_secondary'], 'fontSize': '0.875rem'}),
            dcc.Dropdown(
                id='age-filter',
                options=[{'label': 'All Ages', 'value': 'all'}] + 
                        [{'label': age, 'value': age} for age in sorted([a for a in df['Age_Group'].unique() if pd.notna(a)])],
                value='all',
                clearable=False
            )
        ], width=3)
    ], style={'marginBottom': '2rem'}),

    # --- NEW: RESET BUTTON ROW ---
    dbc.Row([
        dbc.Col([
            dbc.Button(
                "🔄 Reset All Filters", 
                id='btn-reset', 
                color='secondary', 
                outline=True,
                size='sm',  # Small button
                style={
                    'borderColor': COLORS['border'], 
                    'color': COLORS['text_primary'],
                    'backgroundColor': COLORS['surface']
                }
            )
        ], width=12, style={'textAlign': 'right', 'marginBottom': '1rem'}) # Aligned right
    ]),
    
    # KPI Cards Row
    dbc.Row([
        dbc.Col(html.Div(id='kpi-total-students'), width=3),
        dbc.Col(html.Div(id='kpi-avg-support'), width=3),
        dbc.Col(html.Div(id='kpi-high-risk'), width=3),
        dbc.Col(html.Div(id='kpi-support-seeking'), width=3)
    ], style={'marginBottom': '2rem'}),
    
    # Chart 1: Nationality Study Effort (Full Width)
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id='chart-nationality-study', config={'displayModeBar': False})
                ])
            ], style={
                'backgroundColor': COLORS['card'],
                'border': f'1px solid {COLORS["border"]}',
                'borderRadius': '8px'
            })
        ], width=12)
    ], style={'marginBottom': '1.5rem'}),
    
    # Charts 2 & 3: Support Factors + Compensation Matrix (Side by Side)
    dbc.Row([
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id='chart-support-factors', config={'displayModeBar': False})
                ])
            ], style={
                'backgroundColor': COLORS['card'],
                'border': f'1px solid {COLORS["border"]}',
                'borderRadius': '8px'
            })
        ], width=6),
        
        dbc.Col([
            dbc.Card([
                dbc.CardBody([
                    dcc.Graph(id='chart-compensation-matrix', config={'displayModeBar': False})
                ])
            ], style={
                'backgroundColor': COLORS['card'],
                'border': f'1px solid {COLORS["border"]}',
                'borderRadius': '8px'
            })
        ], width=6)
    ], style={'marginBottom': '1.5rem'}),
    
    # Chart 4: Age Attendance Discipline (Full Width)
    dbc.Row([
            dbc.Col([
                dbc.Card([
                    dbc.CardBody([
                        # Graph
                        dcc.Graph(id='chart-age-attendance', config={'displayModeBar': False}),
                        
                        # Slider Control Section (New Location)
                        html.Div([
                            html.Label("🎚️ Adjust Attendance Passing Threshold:", 
                                    style={'color': COLORS['primary'], 'fontWeight': 'bold', 'marginBottom': '10px'}),
                            dcc.Slider(
                                id='global-attendance-slider', # ID stays the same so callback works
                                min=50,
                                max=100,
                                step=5,
                                value=75,
                                marks={i: {'label': f'{i}%', 'style': {'color': COLORS['text_secondary']}} for i in range(50, 101, 10)},
                                tooltip={"placement": "bottom", "always_visible": True}
                            )
                        ], style={'padding': '0px 20px 20px 20px'}), # Padding to separate from graph
                        
                        html.Div(id='attendance-discipline-insights', style={'marginTop': '1rem'})
                    ])
                ], style={
                    'backgroundColor': COLORS['card'],
                    'border': f'1px solid {COLORS["border"]}',
                    'borderRadius': '8px'
                })
            ], width=12)
        ], style={'marginBottom': '2rem'}),
    
    # Footer
    dbc.Row([
        dbc.Col([
            html.Hr(style={'borderColor': COLORS['border']}),
            html.P([
                "CA2 Data Visualization Assignment • ST1502 • ",
                html.Span("Lingger's Dashboard", style={'color': COLORS['primary'], 'fontWeight': '600'}),
                " • AY2526 Sem 2"
            ], style={
                'textAlign': 'center',
                'color': COLORS['text_secondary'],
                'fontSize': '0.875rem',
                'marginTop': '1rem'
            })
        ])
    ])
    
], fluid=True, style={
    'backgroundColor': COLORS['background'],
    'minHeight': '100vh',
    'padding': '2rem'
})

In [405]:
# ============================================================================
# CALLBACKS
# ============================================================================
# Callback 1: Reset Filters
@app.callback(
    [Output('semester-filter', 'value'), Output('course-filter', 'value'),
     Output('nationality-filter', 'value'), Output('age-filter', 'value')],
    [Input('btn-reset', 'n_clicks')],
    prevent_initial_call=True
)
def reset_filters(n):
    return 'all', 'all', 'all', 'all'

# Callback 2: Update Dashboard Components
@app.callback(
    [Output('kpi-total-students', 'children'),
     Output('kpi-avg-support', 'children'),
     Output('kpi-high-risk', 'children'),
     Output('kpi-support-seeking', 'children'),
     Output('chart-nationality-study', 'figure'),
     Output('chart-support-factors', 'figure'),
     Output('chart-compensation-matrix', 'figure'),
     Output('chart-age-attendance', 'figure'),
     Output('attendance-discipline-insights', 'children')],
    [Input('semester-filter', 'value'),
     Input('course-filter', 'value'),
     Input('nationality-filter', 'value'),
     Input('age-filter', 'value'),
     Input('global-attendance-slider', 'value')]
)

def update_dashboard(semester, course, nationality, age_group, attendance_threshold):
    # 1. Base Filter (Semester & Course affect everything)
    df_base = df.copy()
    if semester != 'all':
        df_base = df_base[df_base['PERIOD'] == semester]
    if course != 'all':
        df_base = df_base[df_base['Course_Code'] == course]

    # 2. Fully Filtered Data (For KPIs, Chart 2, Chart 3)
    filtered_df = df_base.copy()
    if nationality != 'all':
        filtered_df = filtered_df[filtered_df['NATIONALITY_STATUS'] == nationality]
    if age_group != 'all':
        filtered_df = filtered_df[filtered_df['Age_Group'] == age_group]

    # 3. Data for Chart 1 (Nationality Box Plot)
    # IGNORES nationality filter, but applies Age filter
    df_chart1 = df_base.copy()
    if age_group != 'all':
        df_chart1 = df_chart1[df_chart1['Age_Group'] == age_group]

    # 4. Data for Chart 4 (Age Risk Bar)
    # IGNORES age filter, but applies Nationality filter
    df_chart4 = df_base.copy()
    if nationality != 'all':
        df_chart4 = df_chart4[df_chart4['NATIONALITY_STATUS'] == nationality]
    
    # --- KPI CALCULATION (FIXED ARGUMENTS) ---
    # We pass semester and course explicitly to match your original function signature
    kpis = calculate_kpis(filtered_df, semester if semester != 'all' else None,
                          course if course != 'all' else None)
    
    kpi1 = create_kpi_card("Total Students", f"{kpis['total_students']:,}", "Unique students analyzed", "👥", 'info')
    kpi2 = create_kpi_card("Avg Support Rating", f"{kpis['avg_support']:.1f}/5", "Across all support factors", "🤝", 'primary')
    kpi3 = create_kpi_card("High Risk %", f"{kpis['high_risk_pct']:.1f}%", "Students with GPA < 2.5", "⚠️", 'danger')
    kpi4 = create_kpi_card("High Support %", f"{kpis['support_seeking_pct']:.1f}%", "Students with support ≥ 4", "🌟", 'success')
    
    # --- GENERATE CHARTS ---
    
    # Chart 1: Uses df_chart1 (All Nationalities)
    chart1 = create_nationality_study_effort(
        df_chart1,
        selected_nationality=None 
    )
    
    # Chart 2: Uses filtered_df (Fully Filtered)
    chart2 = create_support_factors_impact(filtered_df)
    
    # Chart 3: Uses filtered_df (Fully Filtered)
    chart3 = create_attendance_study_compensation_heatmap(filtered_df)
    
    # Chart 4: Uses df_chart4 (All Ages)
    chart4, discipline_stats = create_age_attendance_discipline_slider(df_chart4, attendance_threshold)
    
    # --- INSIGHTS (WITH CRASH PROTECTION) ---
    if not discipline_stats.empty and discipline_stats['Total'].sum() > 0:
        worst_age = discipline_stats.loc[discipline_stats['At_Risk_Pct'].idxmax()]
        best_age = discipline_stats.loc[discipline_stats['At_Risk_Pct'].idxmin()]
        
        insights = dbc.Alert([
            html.H6(f"💡 Attendance Discipline Insights (Threshold: {attendance_threshold}%):", style={'marginBottom': '0.5rem'}),
            html.Ul([
                html.Li(f"🔴 Highest Risk: {worst_age['Age_Group']} ({worst_age['At_Risk_Pct']:.1f}% fail rate)"),
                html.Li(f"🟢 Lowest Risk: {best_age['Age_Group']} ({best_age['At_Risk_Pct']:.1f}% fail rate)"),
                html.Li(f"📊 Overall Impact: {discipline_stats['At_Risk_Count'].sum():.0f} students fail attendance requirements")
            ], style={'marginBottom': 0})
        ], color='info', style={'backgroundColor': COLORS['surface'], 'borderColor': COLORS['info'], 'color': COLORS['text_primary']})
    else:
        # Fallback if filter results in no students
        insights = dbc.Alert("⚠️ No student data available for this filter combination.", color='warning')

    return kpi1, kpi2, kpi3, kpi4, chart1, chart2, chart3, chart4, insights

In [ ]:
# Run the app inside the notebook
# mode='inline' shows it in the output cell
# mode='external' opens it in a new browser tab (better for full-screen dashboards)
# mode='jupyterlab' opens it in a side pane (if using JupyterLab)

if __name__ == '__main__':
    app.run(jupyter_mode='external', port=8051)

Dash app running on http://127.0.0.1:8051/


C:\Users\User\AppData\Local\Temp\ipykernel_38176\3802068129.py:201: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.

C:\Users\User\AppData\Local\Temp\ipykernel_38176\3802068129.py:201: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\User\AppData\Local\Temp\ipykernel_38176\3802068129.py:229: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence